# Pre-generation failure prediction for cost-aware routing, and whether it survives domain shift

**Question 1 (replication, not contribution).** Before a small language model emits a single
token, is it already legible from its hidden states that it will get this query wrong? If so,
a linear probe on those activations can drive *cost-aware routing*: send the query to an
expensive model only when the cheap model is predicted to fail.

**Question 2 (the actual contribution).** Does that signal survive **domain shift**? A router is
trained once and deployed against whatever traffic arrives. Everything below is arranged so that
E4 (full train-on-X / test-on-Y transfer matrix plus a learning curve) and E5 (what happens to a
*single fixed threshold* when the input distribution moves) are the load-bearing results.

---

## Prior art

Pre-generation probing of correctness / difficulty from hidden states is **established work**.
This notebook does not claim to have invented it, and E1-E3 below should be read as a
re-implementation on a smaller model, run only so that E4 and E5 have a calibrated baseline to
sit on top of.

- arXiv:2510.18147
- arXiv:2602.09924
- arXiv:2603.20895
- `KabakaWilliam/llms_know_difficulty` (https://github.com/KabakaWilliam/llms_know_difficulty)

> **Plainly: the probe is not the contribution.** That a linear probe on final-prompt-token
> activations predicts downstream correctness above chance is a known result and is reproduced
> here (E1) only as a control. The contribution claimed by this notebook is E4 and E5: a full
> cross-domain transfer matrix on identical labels with in-domain-only preprocessing, an
> in-domain-sample learning curve quantifying how much target data closes the transfer gap, and
> an explicit account of what a *deployed, fixed* decision threshold does when the domain moves.

---

## What is being compared

Six routers, all scored on **identical labels and identical splits**:

| # | Router | Needs generation first? | Notes |
|---|---|---|---|
| 1 | Random | no | lower bound |
| 2 | Oracle (true labels) | -- | upper bound, not deployable |
| 3 | Prompt length only (logistic regression) | no | **the confound baseline** |
| 4 | Frozen sentence-encoder embedding of the query | no | semantic, but model-agnostic |
| 5 | Post-hoc confidence (answer entropy, max token prob) | **yes** | strictly more expensive than 3 and 4 |
| 6 | Linear probe on weak-model activations | no | the method under test |

Router 5 is included because it is the honest strong baseline, but it is **not cost-comparable**:
it requires the weak model to actually generate before you can decide to escalate, so on every
escalated query you pay the weak model's output cost *and* the strong model's. The cost model in
E3 charges it accordingly.

---

## Label convention (used everywhere)

`y = 1` means **"the weak model will fail on this query"**, i.e. the positive class is
*escalate*. Every AUC below is the AUC for detecting failure. Labels come from the **pass rate
over k sampled generations**, not from a single roll -- see Section 2.

---

## Run cost and hardware

Designed to run top-to-bottom on a single A100. On a Colab T4, reduce `CONFIG["n_examples"]`
and set `CONFIG["run_strong_model"] = False` (E3/E5 then need the fallback described in
Section 6). Cells that touch the network or download weights are flagged in **bold** with a
size estimate.

**Nothing in this notebook fabricates a number.** Every table and figure is produced by the code
in the cell that renders it. Section 9 deliberately leaves the prose as `TODO`.

## Requirements

Install these yourself before running. There are **no `pip install` cells in this notebook**.

```bash
pip install "torch>=2.2" "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" "sentence-transformers>=3.0" "scikit-learn>=1.4" "numpy>=1.26" "pandas>=2.2" "matplotlib>=3.8" "tqdm>=4.66"
```

If you are on CUDA and want the 7B strong model to fit comfortably, also:

```bash
pip install "bitsandbytes>=0.43"
```

Disk: the caches written by Sections 2-3 are a few hundred MB per dataset at the default
`n_examples`. Model weights are separate (see the flagged cells).

## Section 0 - Config, seeding, device

Every hyperparameter lives in `CONFIG` and nowhere else. A run is therefore fully described by
one printable, diffable object, and the on-disk caches in Sections 2-3 are keyed off it, so
changing a knob here invalidates exactly the caches it should and no others.

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import random
import re
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from sklearn.calibration import calibration_curve
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

In [ ]:
CONFIG: Dict[str, Any] = {
    # ---- reproducibility -------------------------------------------------
    "seed": 0,

    # ---- models ----------------------------------------------------------
    # The weak model is referenced ONLY through this constant, so the whole
    # notebook is swappable by editing this one string.
    "model_name": "Qwen/Qwen2.5-0.5B",
    "strong_model_name": "Qwen/Qwen2.5-7B-Instruct",
    "run_strong_model": True,      # set False on a T4; see Section 6 fallback
    "strong_load_in_4bit": False,  # requires bitsandbytes
    "encoder_name": "sentence-transformers/all-MiniLM-L6-v2",

    # ---- data ------------------------------------------------------------
    "datasets": ["gsm8k", "mbpp", "mmlu"],
    "n_examples": {"gsm8k": 1000, "mbpp": 500, "mmlu": 1000},
    "n_fewshot": {"gsm8k": 4, "mbpp": 3, "mmlu": 5},

    # ---- weak-model sampling (label generation) --------------------------
    "k_samples": 8,
    "temperature": 0.7,
    "top_p": 0.95,
    "max_new_tokens": {"gsm8k": 256, "mbpp": 320, "mmlu": 8},
    "gen_batch_size": 8,

    # ---- strong-model reference generations ------------------------------
    "strong_k_samples": 1,       # greedy single roll; see Section 2 note
    "strong_temperature": 0.0,
    "strong_gen_batch_size": 4,

    # ---- labelling -------------------------------------------------------
    # y = 1 ("will fail") iff pass_rate < fail_threshold.
    "fail_threshold": 0.5,

    # ---- activations -----------------------------------------------------
    "act_batch_size": 16,
    "act_max_prompt_tokens": 1024,

    # ---- post-hoc confidence --------------------------------------------
    "posthoc_sample_index": 0,   # which of the k generations to score
    "posthoc_batch_size": 2,

    # ---- probe / model selection ----------------------------------------
    "layer_grid_stride": 2,      # candidate layers = range(0, L+1, stride) + [L]
    "probe_C_grid": [0.01, 0.1, 1.0],
    "probe_max_iter": 2000,
    "pca_components": None,      # None = no PCA; int = fit on train fold only

    # ---- resampling ------------------------------------------------------
    "n_outer_splits": 50,        # every reported AUC is a mean over these
    "outer_test_size": 0.3,
    "inner_folds": 5,
    "ci_alpha": 0.05,

    # ---- routing / cost --------------------------------------------------
    "escalation_report_points": [0.20, 0.30, 0.50],
    "target_escalation_rate": 0.30,
    "frontier_grid": 41,         # number of escalation fractions on the curve
    "calibration_bins": 10,
    # USD per million tokens. These are placeholders for the *ratio*; the cost
    # axis is only ever interpreted relatively. See Section 6.
    "price_per_mtok": {
        "weak":   {"in": 0.05, "out": 0.10},
        "strong": {"in": 0.60, "out": 1.80},
    },

    # ---- E4 learning curve ----------------------------------------------
    "learning_curve_sizes": [0, 25, 50, 100, 200, 400],
    "learning_curve_reps": 20,

    # ---- sanity thresholds ----------------------------------------------
    "pass_rate_flag_low": 0.10,
    "pass_rate_flag_high": 0.90,

    # ---- execution sandbox (MBPP grading) --------------------------------
    "mbpp_exec_timeout_s": 10,

    # ---- io --------------------------------------------------------------
    "cache_dir": "cache",
    "fig_dir": "figures",
    "device": None,              # None = auto (cuda > mps > cpu)
}

print(json.dumps(CONFIG, indent=2, default=str))

In [ ]:
def set_seed(seed: int) -> None:
    """Seed every RNG this notebook can reach.

    Args:
        seed: Non-negative integer seed applied to `random`, NumPy, and torch
            (CPU and all CUDA devices).

    Returns:
        None.

    Raises:
        ValueError: If `seed` is negative.
    """
    if seed < 0:
        raise ValueError(f"seed must be non-negative, got {seed}")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def select_device(requested: Optional[str] = None) -> torch.device:
    """Pick a torch device, preferring CUDA, then Apple MPS, then CPU.

    Args:
        requested: Explicit device string (e.g. "cuda", "mps", "cpu"). If None,
            the best available device is chosen automatically.

    Returns:
        The selected `torch.device`.

    Raises:
        RuntimeError: If `requested` names a backend that is not available.
    """
    if requested is not None:
        dev = torch.device(requested)
        if dev.type == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but not available")
        if dev.type == "mps" and not torch.backends.mps.is_available():
            raise RuntimeError("MPS requested but not available")
        return dev
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


set_seed(CONFIG["seed"])
DEVICE = select_device(CONFIG["device"])
CACHE_DIR = Path(CONFIG["cache_dir"])
FIG_DIR = Path(CONFIG["fig_dir"])
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("device      :", DEVICE)
print("torch       :", torch.__version__)
print("cache dir   :", CACHE_DIR.resolve())
print("figure dir  :", FIG_DIR.resolve())

In [ ]:
def cache_key(**parts: Any) -> str:
    """Build a short, deterministic cache key from keyword parts.

    The key is a readable prefix (the sorted values, sanitised) followed by an
    8-character hash of the full JSON encoding, so two calls that differ in any
    part -- including parts not visible in the prefix -- get different keys.

    Args:
        **parts: Arbitrary JSON-serialisable values identifying the artefact,
            e.g. `model="Qwen/Qwen2.5-0.5B", dataset="gsm8k", k=8`.

    Returns:
        A filesystem-safe key string.

    Raises:
        TypeError: If any part is not JSON-serialisable.
    """
    blob = json.dumps(parts, sort_keys=True, default=str)
    digest = hashlib.sha1(blob.encode("utf-8")).hexdigest()[:8]
    readable = "_".join(
        re.sub(r"[^A-Za-z0-9.-]+", "-", str(parts[k]))[:24] for k in sorted(parts)
    )
    return f"{readable}__{digest}"


def cache_json_path(kind: str, **parts: Any) -> Path:
    """Return the on-disk path for a JSON cache artefact.

    Args:
        kind: Artefact family, used as the filename prefix (e.g. "gens").
        **parts: Cache-key parts, forwarded to `cache_key`.

    Returns:
        Path under `CACHE_DIR` ending in `.json`. The file may not exist.

    Raises:
        TypeError: If any part is not JSON-serialisable.
    """
    return CACHE_DIR / f"{kind}__{cache_key(**parts)}.json"


def cache_npz_path(kind: str, **parts: Any) -> Path:
    """Return the on-disk path for a NumPy `.npz` cache artefact.

    Args:
        kind: Artefact family, used as the filename prefix (e.g. "acts").
        **parts: Cache-key parts, forwarded to `cache_key`.

    Returns:
        Path under `CACHE_DIR` ending in `.npz`. The file may not exist.

    Raises:
        TypeError: If any part is not JSON-serialisable.
    """
    return CACHE_DIR / f"{kind}__{cache_key(**parts)}.npz"


def read_json(path: Path) -> Any:
    """Load a UTF-8 JSON file.

    Args:
        path: File to read.

    Returns:
        The decoded object.

    Raises:
        FileNotFoundError: If `path` does not exist.
        json.JSONDecodeError: If the file is not valid JSON.
    """
    with path.open("r", encoding="utf-8") as fh:
        return json.load(fh)


def write_json(path: Path, obj: Any) -> None:
    """Write an object as UTF-8 JSON, creating parent directories as needed.

    The write is atomic (temp file plus rename) so an interrupted run cannot
    leave a half-written cache that a later run would silently trust.

    Args:
        path: Destination file.
        obj: JSON-serialisable object.

    Returns:
        None.

    Raises:
        TypeError: If `obj` is not JSON-serialisable.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as fh:
        json.dump(obj, fh)
    tmp.replace(path)


def free_memory(*objs: Any) -> None:
    """Delete references, run the garbage collector, and empty the CUDA cache.

    Args:
        *objs: Objects to drop references to before collecting.

    Returns:
        None.

    Raises:
        Nothing.
    """
    for _ in objs:
        pass
    del objs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Section 1 - Data loading

Three datasets chosen because they stress different capabilities, which is the whole point of E4:

| Dataset | Skill | Grading |
|---|---|---|
| GSM8K | multi-step arithmetic reasoning | exact match on the final number |
| MBPP | Python code synthesis | **executed** against the provided unit tests |
| MMLU | factual recall / multiple choice | exact match on the answer letter |

Every record is normalised to one schema so that Sections 2-8 never branch on dataset except
inside the prompt builder and the grader:

```
{"qid": str, "dataset": str, "prompt": str, "reference": Any, "meta": dict}
```

**Few-shot prefixes.** A 0.5B *base* model answers zero-shot almost never, which would drive the
pass rate under the 0.1 floor and leave no routing signal to detect. Each dataset therefore gets
a fixed, seeded few-shot prefix drawn from the *train* split, held constant across every test
query. Held constant matters twice: it keeps the label distribution usable, and it means the
prompt-length variation the E2 confound analysis operates on comes from the *query*, not from
the scaffold.

> **Contestable choice: few-shot count.** More shots raises the pass rate and shrinks the
> positive class; fewer shots does the opposite. `n_fewshot` is per-dataset in `CONFIG` and the
> Section 2 sanity check exists precisely to catch a setting that has pushed a dataset into a
> degenerate label regime. Alternative: use an instruct-tuned weak model zero-shot
> (`Qwen2.5-0.5B-Instruct`), which removes the prefix entirely but changes what "the cheap model"
> means and makes the length confound stronger, since prompt length then varies only with query
> length.

**The label-quality sanity check is *defined* here and *called* in Section 2**, because it needs
labels that do not exist until generation has run. Keeping the call in Section 2 is what lets
the notebook execute top-to-bottom.

> 🌐 **Network cell.** Downloads GSM8K (~5 MB), MBPP (~2 MB) and MMLU (~170 MB for the `all`
> config) from the Hugging Face Hub into `~/.cache/huggingface`. No model weights yet.

In [ ]:
from datasets import load_dataset


def _norm_ws(text: str) -> str:
    """Collapse runs of whitespace and strip, leaving a single-space-joined string.

    Args:
        text: Raw string.

    Returns:
        The whitespace-normalised string.

    Raises:
        Nothing.
    """
    return re.sub(r"\s+", " ", text).strip()


def load_gsm8k(n: int, n_fewshot: int, seed: int) -> Tuple[List[Dict[str, Any]], str]:
    """Load GSM8K test questions plus a fixed few-shot prefix from the train split.

    Args:
        n: Number of test records to keep (sampled without replacement).
        n_fewshot: Number of worked examples to place in the prefix.
        seed: Seed controlling both the prefix draw and the test subsample.

    Returns:
        `(records, prefix)` where `records` is a list of normalised record dicts
        with `reference` set to the gold numeric answer string, and `prefix` is
        the few-shot text prepended to every prompt.

    Raises:
        ValueError: If `n` or `n_fewshot` exceeds the available split size.
    """
    train = load_dataset("openai/gsm8k", "main", split="train")
    test = load_dataset("openai/gsm8k", "main", split="test")
    if n_fewshot > len(train):
        raise ValueError(f"n_fewshot={n_fewshot} exceeds train split of {len(train)}")
    if n > len(test):
        raise ValueError(f"n={n} exceeds test split of {len(test)}")

    rng = np.random.default_rng(seed)
    shot_idx = rng.choice(len(train), size=n_fewshot, replace=False)
    shots = []
    for i in shot_idx:
        ex = train[int(i)]
        sol = re.sub(r"<<.*?>>", "", ex["answer"]).strip()
        shots.append(f"Question: {ex['question'].strip()}\nAnswer: {sol}")
    prefix = "\n\n".join(shots) + ("\n\n" if shots else "")

    test_idx = rng.choice(len(test), size=n, replace=False)
    records = []
    for i in sorted(int(j) for j in test_idx):
        ex = test[i]
        gold = ex["answer"].split("####")[-1].strip().replace(",", "")
        records.append({
            "qid": f"gsm8k-{i}",
            "dataset": "gsm8k",
            "prompt": f"{prefix}Question: {ex['question'].strip()}\nAnswer:",
            "reference": gold,
            "meta": {"question": ex["question"].strip()},
        })
    return records, prefix


def load_mbpp(n: int, n_fewshot: int, seed: int) -> Tuple[List[Dict[str, Any]], str]:
    """Load MBPP problems plus a fixed few-shot prefix, with unit tests as the reference.

    Args:
        n: Number of test records to keep (sampled without replacement).
        n_fewshot: Number of worked examples in the prefix, drawn from the
            MBPP `prompt` split (the canonical few-shot pool, ids 1-10).
        seed: Seed controlling both the prefix draw and the test subsample.

    Returns:
        `(records, prefix)` where each record's `reference` is
        `{"test_list": [...], "test_setup_code": str}` used by the executing grader.

    Raises:
        ValueError: If `n` or `n_fewshot` exceeds the available split size.
    """
    shot_pool = load_dataset("google-research-datasets/mbpp", "full", split="prompt")
    test = load_dataset("google-research-datasets/mbpp", "full", split="test")
    if n_fewshot > len(shot_pool):
        raise ValueError(f"n_fewshot={n_fewshot} exceeds prompt split of {len(shot_pool)}")
    if n > len(test):
        raise ValueError(f"n={n} exceeds test split of {len(test)}")

    rng = np.random.default_rng(seed)
    shot_idx = rng.choice(len(shot_pool), size=n_fewshot, replace=False)
    shots = []
    for i in shot_idx:
        ex = shot_pool[int(i)]
        tests = "\n".join(ex["test_list"])
        shots.append(
            f"You are an expert Python programmer.\n{ex['text'].strip()}\n"
            f"Your code should pass these tests:\n{tests}\n"
            f"[BEGIN]\n{ex['code'].strip()}\n[DONE]"
        )
    prefix = "\n\n".join(shots) + ("\n\n" if shots else "")

    test_idx = rng.choice(len(test), size=n, replace=False)
    records = []
    for i in sorted(int(j) for j in test_idx):
        ex = test[i]
        tests = "\n".join(ex["test_list"])
        records.append({
            "qid": f"mbpp-{ex['task_id']}",
            "dataset": "mbpp",
            "prompt": (
                f"{prefix}You are an expert Python programmer.\n{ex['text'].strip()}\n"
                f"Your code should pass these tests:\n{tests}\n[BEGIN]\n"
            ),
            "reference": {
                "test_list": list(ex["test_list"]),
                "test_setup_code": ex.get("test_setup_code", "") or "",
            },
            "meta": {"question": ex["text"].strip()},
        })
    return records, prefix


def load_mmlu(n: int, n_fewshot: int, seed: int) -> Tuple[List[Dict[str, Any]], str]:
    """Load MMLU questions plus a fixed few-shot prefix from the dev split.

    Args:
        n: Number of test records to keep (sampled without replacement).
        n_fewshot: Number of worked examples in the prefix, drawn from `dev`.
        seed: Seed controlling both the prefix draw and the test subsample.

    Returns:
        `(records, prefix)` where each record's `reference` is the gold answer
        letter in "A".."D".

    Raises:
        ValueError: If `n` or `n_fewshot` exceeds the available split size.
    """
    dev = load_dataset("cais/mmlu", "all", split="dev")
    test = load_dataset("cais/mmlu", "all", split="test")
    if n_fewshot > len(dev):
        raise ValueError(f"n_fewshot={n_fewshot} exceeds dev split of {len(dev)}")
    if n > len(test):
        raise ValueError(f"n={n} exceeds test split of {len(test)}")

    letters = ["A", "B", "C", "D"]

    def render(ex: Dict[str, Any]) -> str:
        opts = "\n".join(f"{letters[j]}. {c}" for j, c in enumerate(ex["choices"]))
        return f"Question: {_norm_ws(ex['question'])}\n{opts}\nAnswer:"

    rng = np.random.default_rng(seed)
    shot_idx = rng.choice(len(dev), size=n_fewshot, replace=False)
    shots = [
        f"{render(dev[int(i)])} {letters[int(dev[int(i)]['answer'])]}" for i in shot_idx
    ]
    prefix = (
        "The following are multiple choice questions. Answer with a single letter.\n\n"
        + "\n\n".join(shots)
        + ("\n\n" if shots else "")
    )

    test_idx = rng.choice(len(test), size=n, replace=False)
    records = []
    for i in sorted(int(j) for j in test_idx):
        ex = test[i]
        records.append({
            "qid": f"mmlu-{i}",
            "dataset": "mmlu",
            "prompt": prefix + render(ex),
            "reference": letters[int(ex["answer"])],
            "meta": {"question": _norm_ws(ex["question"]), "subject": ex.get("subject", "")},
        })
    return records, prefix


LOADERS: Dict[str, Callable[[int, int, int], Tuple[List[Dict[str, Any]], str]]] = {
    "gsm8k": load_gsm8k,
    "mbpp": load_mbpp,
    "mmlu": load_mmlu,
}

In [ ]:
RECORDS: Dict[str, List[Dict[str, Any]]] = {}
PREFIXES: Dict[str, str] = {}

for _ds in CONFIG["datasets"]:
    _recs, _pref = LOADERS[_ds](
        CONFIG["n_examples"][_ds], CONFIG["n_fewshot"][_ds], CONFIG["seed"]
    )
    RECORDS[_ds] = _recs
    PREFIXES[_ds] = _pref
    print(f"{_ds:6s}  n={len(_recs):5d}  prefix_chars={len(_pref):6d}")

print("\n--- one example prompt per dataset (truncated) ---")
for _ds in CONFIG["datasets"]:
    print(f"\n===== {_ds} =====")
    print(RECORDS[_ds][0]["prompt"][-800:])

In [ ]:
def check_label_quality(
    pass_rates: Dict[str, np.ndarray],
    labels: Dict[str, np.ndarray],
    low: float,
    high: float,
) -> pd.DataFrame:
    """Report weak-model pass rate per dataset and flag degenerate label regimes.

    A dataset whose mean pass rate sits below `low` or above `high` carries
    almost no routing signal: nearly every query is a failure (or nearly none
    is), so a router has nothing to discriminate and any AUC reported on it is
    computed over a handful of minority-class points.

    Args:
        pass_rates: Mapping dataset -> array of per-query pass rates in [0, 1].
        labels: Mapping dataset -> array of binary labels (1 = will fail).
        low: Lower flag threshold on the mean pass rate.
        high: Upper flag threshold on the mean pass rate.

    Returns:
        A DataFrame indexed by dataset with columns `n`, `mean_pass_rate`,
        `fail_rate`, `n_fail`, `n_pass`, `frac_all_pass`, `frac_all_fail`,
        and `FLAG`. `FLAG` is the empty string when the dataset is usable.

    Raises:
        KeyError: If `pass_rates` and `labels` do not cover the same datasets.
        ValueError: If any pass rate falls outside [0, 1].
    """
    if set(pass_rates) != set(labels):
        raise KeyError("pass_rates and labels must cover identical datasets")
    rows = []
    for ds in sorted(pass_rates):
        pr = np.asarray(pass_rates[ds], dtype=float)
        y = np.asarray(labels[ds], dtype=int)
        if pr.size and (pr.min() < 0.0 or pr.max() > 1.0):
            raise ValueError(f"pass rates for {ds} outside [0, 1]")
        mean_pr = float(pr.mean()) if pr.size else float("nan")
        flags = []
        if mean_pr < low:
            flags.append(f"PASS RATE < {low:.2f}: weak model almost never solves this")
        if mean_pr > high:
            flags.append(f"PASS RATE > {high:.2f}: weak model almost always solves this")
        minority = min(int(y.sum()), int((1 - y).sum()))
        if minority < 30:
            flags.append(f"MINORITY CLASS n={minority} < 30: AUC will be very noisy")
        rows.append({
            "dataset": ds,
            "n": int(pr.size),
            "mean_pass_rate": mean_pr,
            "fail_rate": float(y.mean()) if y.size else float("nan"),
            "n_fail": int(y.sum()),
            "n_pass": int((1 - y).sum()),
            "frac_all_pass": float((pr == 1.0).mean()) if pr.size else float("nan"),
            "frac_all_fail": float((pr == 0.0).mean()) if pr.size else float("nan"),
            "FLAG": " | ".join(flags),
        })
    return pd.DataFrame(rows).set_index("dataset")


print("check_label_quality defined; it is CALLED in Section 2, once labels exist.")

## Section 2 - Generation and labelling

**Labels come from a pass rate, not a single roll.** For each query we sample `k = 8` generations
at temperature 0.7 and record how many are correct. The binary label is
`y = 1 iff pass_rate < fail_threshold` (default 0.5). A single greedy roll would make the label a
sample from a Bernoulli whose parameter is what we actually care about; the resulting label noise
is irreducible and would cap every AUC in the notebook at an unknown ceiling. Averaging over
k rolls shrinks that noise by roughly `sqrt(k)`.

> **Contestable choice: k and temperature.** `k = 8` at `T = 0.7` is the usual pass@k sampling
> configuration and costs 8x a greedy pass. `k = 1` greedy is 8x cheaper but reintroduces the
> label noise just described; `k = 32` would halve the residual noise again for 4x the compute,
> which is not worth it when the probe's own variance across splits is larger. `T = 0.7` rather
> than `T = 1.0` because the point is to estimate *typical deployed* behaviour, and most serving
> stacks sample below 1.0; rather than `T = 0` because a deterministic decode collapses the pass
> rate to {0, 1} and throws away the graded difficulty signal that makes the label useful.
>
> **Contestable choice: `fail_threshold = 0.5`.** This binarises a genuinely continuous quantity.
> The raw pass rates are cached, so the threshold can be swept without re-running the model --
> Section 9 leaves a hook for exactly that. An alternative is to skip binarisation and regress the
> pass rate directly, then threshold the prediction; that is arguably cleaner but makes AUC
> and the oracle/random baselines harder to state, so it is not the primary analysis here.

**Raw generations are cached to disk** keyed by `(model, dataset, k, temperature, top_p,
max_new_tokens, n_examples, n_fewshot, seed)`. Labels are recomputed from that cache on every
run, so changing the grader or the threshold never requires touching the GPU again.

### A note on executing model-generated code

The MBPP grader **runs generated Python**. It runs in a separate process with a wall-clock
timeout and no arguments, but it is not a real sandbox: this code is generated by a 0.5B model
from a public benchmark, not adversarial input, and that is the only reason this is acceptable.
Do not point this grader at untrusted generations.

> 🌐 **Network cell + model download.** Fetches `CONFIG["model_name"]`
> (`Qwen/Qwen2.5-0.5B`, ~1 GB in bf16/fp32 safetensors) from the Hub on first run.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_lm(model_name: str, device: torch.device, load_in_4bit: bool = False):
    """Load a causal LM and its tokenizer, configured for left-padded batch generation.

    Left padding is required: with right padding a decoder-only model's last
    position is a pad token, which would corrupt both generation and the
    final-prompt-token activations extracted in Section 3.

    Args:
        model_name: Hugging Face repo id.
        device: Device to place the model on. Ignored when `load_in_4bit` is
            True, since `accelerate` handles placement.
        load_in_4bit: Load 4-bit quantised weights via bitsandbytes.

    Returns:
        `(model, tokenizer)` with the model in eval mode and gradients disabled.

    Raises:
        ImportError: If `load_in_4bit` is True and bitsandbytes is unavailable.
    """
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    kwargs: Dict[str, Any] = {"torch_dtype": torch.float32}
    if device.type == "cuda":
        kwargs["torch_dtype"] = torch.bfloat16
    if load_in_4bit:
        try:
            import bitsandbytes  # noqa: F401
        except ImportError as exc:
            raise ImportError("load_in_4bit=True requires bitsandbytes") from exc
        kwargs.update({"load_in_4bit": True, "device_map": "auto"})

    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    if not load_in_4bit:
        model = model.to(device)
    model.eval()
    model.requires_grad_(False)
    return model, tok


@torch.no_grad()
def generate_samples(
    model,
    tok,
    prompts: Sequence[str],
    k: int,
    temperature: float,
    top_p: float,
    max_new_tokens: int,
    batch_size: int,
    seed: int,
    verbose: bool = True,
) -> List[List[str]]:
    """Sample `k` completions for each prompt.

    Args:
        model: A causal LM in eval mode.
        tok: Its tokenizer, with `padding_side == "left"`.
        prompts: Prompt strings.
        k: Completions per prompt. `k == 1` with `temperature == 0` decodes greedily.
        temperature: Sampling temperature; 0 selects greedy decoding.
        top_p: Nucleus sampling mass; ignored under greedy decoding.
        max_new_tokens: Hard cap on generated tokens per completion.
        batch_size: Number of *prompts* per forward batch (each expands to `k`
            sequences internally).
        seed: Seed re-applied before the loop so a cache miss reproduces exactly.
        verbose: Print progress every batch.

    Returns:
        A list of length `len(prompts)`, each a list of `k` completion strings
        with the prompt stripped off.

    Raises:
        ValueError: If `k < 1` or `max_new_tokens < 1`.
    """
    if k < 1:
        raise ValueError(f"k must be >= 1, got {k}")
    if max_new_tokens < 1:
        raise ValueError(f"max_new_tokens must be >= 1, got {max_new_tokens}")

    set_seed(seed)
    device = next(model.parameters()).device
    greedy = temperature <= 0.0
    out: List[List[str]] = []
    t0 = time.time()

    for start in range(0, len(prompts), batch_size):
        chunk = list(prompts[start : start + batch_size])
        enc = tok(chunk, return_tensors="pt", padding=True, truncation=False).to(device)
        gen_kwargs: Dict[str, Any] = {
            "max_new_tokens": max_new_tokens,
            "num_return_sequences": k,
            "pad_token_id": tok.pad_token_id,
            "eos_token_id": tok.eos_token_id,
        }
        if greedy:
            gen_kwargs["do_sample"] = False
        else:
            gen_kwargs.update({"do_sample": True, "temperature": temperature, "top_p": top_p})

        seqs = model.generate(**enc, **gen_kwargs)
        new_tokens = seqs[:, enc["input_ids"].shape[1] :]
        texts = tok.batch_decode(new_tokens, skip_special_tokens=True)
        for i in range(len(chunk)):
            out.append(texts[i * k : (i + 1) * k])

        if verbose:
            done = min(start + batch_size, len(prompts))
            rate = done / max(time.time() - t0, 1e-9)
            print(f"  generated {done}/{len(prompts)} prompts  ({rate:.2f} prompt/s)", flush=True)

    return out

In [ ]:
_NUM_RE = re.compile(r"-?\d[\d,]*\.?\d*")


def _to_float(text: str) -> Optional[float]:
    """Parse a possibly comma-grouped numeric string.

    Args:
        text: Candidate number, e.g. "1,234.50".

    Returns:
        The float value, or None if it does not parse.

    Raises:
        Nothing.
    """
    try:
        return float(text.replace(",", "").rstrip("."))
    except ValueError:
        return None


def grade_gsm8k(completion: str, reference: Any) -> bool:
    """Grade a GSM8K completion by comparing its final number to the gold answer.

    The completion is first truncated at the start of a hallucinated next
    question, since a base model continues the few-shot pattern past its answer.

    Args:
        completion: Generated text (prompt already stripped).
        reference: Gold answer string, e.g. "72".

    Returns:
        True if the last number in the completion equals the gold answer to
        within 1e-4 relative tolerance.

    Raises:
        Nothing.
    """
    body = re.split(r"\nQuestion:|\nQ:", completion)[0]
    nums = _NUM_RE.findall(body)
    if not nums:
        return False
    pred = _to_float(nums[-1])
    gold = _to_float(str(reference))
    if pred is None or gold is None:
        return False
    return bool(abs(pred - gold) <= 1e-4 * max(1.0, abs(gold)))


def grade_mmlu(completion: str, reference: Any) -> bool:
    """Grade an MMLU completion by its first emitted answer letter.

    Args:
        completion: Generated text (prompt already stripped).
        reference: Gold letter in "A".."D".

    Returns:
        True if the first A-D token in the completion matches the gold letter.

    Raises:
        Nothing.
    """
    m = re.search(r"\b([ABCD])\b", completion.strip().upper())
    return bool(m and m.group(1) == str(reference).strip().upper())


def extract_mbpp_code(completion: str) -> str:
    """Cut an MBPP completion down to the code block the few-shot format asks for.

    Args:
        completion: Generated text (prompt already stripped).

    Returns:
        The text up to the first `[DONE]` marker, with any stray markdown fence
        removed. Returns the whole completion if no marker is present.

    Raises:
        Nothing.
    """
    body = completion.split("[DONE]")[0]
    body = re.split(r"\n\s*You are an expert Python programmer", body)[0]
    body = body.replace("```python", "").replace("```", "")
    return body.strip()


def grade_mbpp(completion: str, reference: Any, timeout_s: int) -> bool:
    """Grade an MBPP completion by executing it against the provided unit tests.

    Runs in a fresh interpreter process with a wall-clock timeout. This is
    isolation, not sandboxing -- see the section note.

    Args:
        completion: Generated text (prompt already stripped).
        reference: Dict with keys `test_list` (list of assert statements) and
            `test_setup_code` (string, possibly empty).
        timeout_s: Wall-clock limit for the child process, in seconds.

    Returns:
        True if the child exits 0, meaning every assertion passed.

    Raises:
        KeyError: If `reference` lacks `test_list`.
    """
    code_body = extract_mbpp_code(completion)
    if not code_body:
        return False
    setup = reference.get("test_setup_code", "") or ""
    tests = "\n".join(reference["test_list"])
    script = f"{code_body}\n\n{setup}\n\n{tests}\n"

    with tempfile.TemporaryDirectory() as tmpdir:
        path = Path(tmpdir) / "candidate.py"
        path.write_text(script, encoding="utf-8")
        try:
            proc = subprocess.run(
                [sys.executable, str(path)],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
                stdin=subprocess.DEVNULL,
                timeout=timeout_s,
                cwd=tmpdir,
            )
        except subprocess.TimeoutExpired:
            return False
        except OSError:
            return False
    return proc.returncode == 0


def grade(dataset: str, completion: str, reference: Any) -> bool:
    """Dispatch to the grader for a dataset.

    Args:
        dataset: One of "gsm8k", "mbpp", "mmlu".
        completion: Generated text (prompt already stripped).
        reference: Dataset-specific gold value.

    Returns:
        True if the completion is correct.

    Raises:
        KeyError: If `dataset` has no registered grader.
    """
    if dataset == "gsm8k":
        return grade_gsm8k(completion, reference)
    if dataset == "mmlu":
        return grade_mmlu(completion, reference)
    if dataset == "mbpp":
        return grade_mbpp(completion, reference, CONFIG["mbpp_exec_timeout_s"])
    raise KeyError(f"no grader registered for dataset {dataset!r}")

In [ ]:
def generation_cache_path(role: str, dataset: str) -> Path:
    """Path of the raw-generation cache for one (role, dataset) pair.

    The key includes every knob that changes the generations, so a config edit
    that matters invalidates the cache and one that does not, does not.

    Args:
        role: Either "weak" or "strong".
        dataset: Dataset name.

    Returns:
        Path to the `.json` cache file, which may not exist.

    Raises:
        ValueError: If `role` is not "weak" or "strong".
    """
    if role == "weak":
        return cache_json_path(
            "gens",
            role=role,
            model=CONFIG["model_name"],
            dataset=dataset,
            k=CONFIG["k_samples"],
            temperature=CONFIG["temperature"],
            top_p=CONFIG["top_p"],
            max_new_tokens=CONFIG["max_new_tokens"][dataset],
            n=CONFIG["n_examples"][dataset],
            n_fewshot=CONFIG["n_fewshot"][dataset],
            seed=CONFIG["seed"],
        )
    if role == "strong":
        return cache_json_path(
            "gens",
            role=role,
            model=CONFIG["strong_model_name"],
            dataset=dataset,
            k=CONFIG["strong_k_samples"],
            temperature=CONFIG["strong_temperature"],
            top_p=CONFIG["top_p"],
            max_new_tokens=CONFIG["max_new_tokens"][dataset],
            n=CONFIG["n_examples"][dataset],
            n_fewshot=CONFIG["n_fewshot"][dataset],
            seed=CONFIG["seed"],
        )
    raise ValueError(f"role must be 'weak' or 'strong', got {role!r}")


def ensure_generations(role: str, dataset: str, model, tok) -> Dict[str, Any]:
    """Return cached raw generations for a dataset, producing them if absent.

    Args:
        role: "weak" or "strong"; selects which sampling config applies.
        dataset: Dataset name; records are taken from the global `RECORDS`.
        model: Loaded causal LM, or None if the cache is expected to hit.
        tok: The model's tokenizer, or None if the cache is expected to hit.

    Returns:
        Dict with keys `qids` (list[str]) and `generations`
        (list[list[str]], outer length = n queries, inner = k).

    Raises:
        RuntimeError: If the cache misses and `model` is None.
    """
    path = generation_cache_path(role, dataset)
    if path.exists():
        print(f"[cache hit ] {role}/{dataset}  {path.name}")
        return read_json(path)
    if model is None:
        raise RuntimeError(
            f"generation cache miss for {role}/{dataset} at {path} and no model was supplied"
        )

    print(f"[cache miss] {role}/{dataset}  generating -> {path.name}")
    recs = RECORDS[dataset]
    prompts = [r["prompt"] for r in recs]
    k = CONFIG["k_samples"] if role == "weak" else CONFIG["strong_k_samples"]
    temp = CONFIG["temperature"] if role == "weak" else CONFIG["strong_temperature"]
    bs = CONFIG["gen_batch_size"] if role == "weak" else CONFIG["strong_gen_batch_size"]

    gens = generate_samples(
        model, tok, prompts,
        k=k,
        temperature=temp,
        top_p=CONFIG["top_p"],
        max_new_tokens=CONFIG["max_new_tokens"][dataset],
        batch_size=bs,
        seed=CONFIG["seed"],
    )
    payload = {"qids": [r["qid"] for r in recs], "generations": gens}
    write_json(path, payload)
    return payload


def label_from_generations(
    dataset: str, payload: Dict[str, Any], fail_threshold: float
) -> Dict[str, np.ndarray]:
    """Grade cached generations and derive pass rates, labels and output lengths.

    Grading is pure post-processing over the cache, so the threshold or the
    grader can change without re-running any model.

    Args:
        dataset: Dataset name, used to pick the grader and the gold references.
        payload: Output of `ensure_generations` for this dataset.
        fail_threshold: Pass rates strictly below this become `y = 1` (fail).

    Returns:
        Dict with arrays `pass_rate` (float, n), `y` (int, n), `n_correct`
        (int, n), `k` (int, n) and `median_out_chars` (float, n).

    Raises:
        ValueError: If the cached qids do not match the loaded records.
    """
    recs = RECORDS[dataset]
    if payload["qids"] != [r["qid"] for r in recs]:
        raise ValueError(
            f"cached qids for {dataset} do not match loaded records; delete the cache and rerun"
        )
    ref_by_qid = {r["qid"]: r["reference"] for r in recs}

    pass_rate, n_correct, ks, out_len = [], [], [], []
    for qid, gens in zip(payload["qids"], payload["generations"]):
        oks = [grade(dataset, g, ref_by_qid[qid]) for g in gens]
        n_correct.append(int(sum(oks)))
        ks.append(len(oks))
        pass_rate.append(float(sum(oks)) / max(len(oks), 1))
        out_len.append(float(np.median([len(g) for g in gens])) if gens else 0.0)

    pr = np.asarray(pass_rate, dtype=float)
    return {
        "pass_rate": pr,
        "y": (pr < fail_threshold).astype(int),
        "n_correct": np.asarray(n_correct, dtype=int),
        "k": np.asarray(ks, dtype=int),
        "median_out_chars": np.asarray(out_len, dtype=float),
    }

> 🌐 **Model download + heavy compute.** First run downloads `Qwen/Qwen2.5-0.5B` (~1 GB) and
> samples `k = 8` generations for every query in every dataset. On an A100 this is roughly
> 20-40 min at the default `n_examples`; on a T4, cut `n_examples` first. Subsequent runs read
> the JSON cache and skip the GPU entirely.

In [ ]:
WEAK_MODEL, WEAK_TOK = load_lm(CONFIG["model_name"], DEVICE)
print(f"loaded {CONFIG['model_name']}  n_layers={WEAK_MODEL.config.num_hidden_layers}  "
      f"hidden={WEAK_MODEL.config.hidden_size}  vocab={WEAK_MODEL.config.vocab_size}")

WEAK_GENS: Dict[str, Dict[str, Any]] = {}
for _ds in CONFIG["datasets"]:
    WEAK_GENS[_ds] = ensure_generations("weak", _ds, WEAK_MODEL, WEAK_TOK)

In [ ]:
LABELS: Dict[str, Dict[str, np.ndarray]] = {
    ds: label_from_generations(ds, WEAK_GENS[ds], CONFIG["fail_threshold"])
    for ds in CONFIG["datasets"]
}

quality = check_label_quality(
    {ds: LABELS[ds]["pass_rate"] for ds in CONFIG["datasets"]},
    {ds: LABELS[ds]["y"] for ds in CONFIG["datasets"]},
    CONFIG["pass_rate_flag_low"],
    CONFIG["pass_rate_flag_high"],
)
display(quality)

_flagged = quality.index[quality["FLAG"] != ""].tolist()
if _flagged:
    print("\n!!! FLAGGED DATASETS:", _flagged)
    print("Every downstream AUC for these is suspect. Fix by adjusting n_fewshot,")
    print("max_new_tokens, or fail_threshold in CONFIG -- then rerun from Section 1.")
else:
    print("\nAll datasets inside the usable pass-rate band; labels carry routing signal.")

In [ ]:
fig, axes = plt.subplots(1, len(CONFIG["datasets"]), figsize=(4.2 * len(CONFIG["datasets"]), 3.2))
axes = np.atleast_1d(axes)
for ax, ds in zip(axes, CONFIG["datasets"]):
    pr = LABELS[ds]["pass_rate"]
    k = int(LABELS[ds]["k"].max())
    ax.hist(pr, bins=np.linspace(-0.5 / k, 1 + 0.5 / k, k + 2), edgecolor="black", linewidth=0.5)
    ax.axvline(CONFIG["fail_threshold"], color="crimson", linestyle="--",
               label=f"fail_threshold={CONFIG['fail_threshold']}")
    ax.set_title(f"{ds}  (mean={pr.mean():.3f})")
    ax.set_xlabel(f"pass rate over k={k}")
    ax.set_ylabel("queries")
    ax.legend(fontsize=8)
fig.suptitle("Weak-model pass-rate distribution -- the raw material of every label below", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "s2_pass_rate_hist.png", dpi=150, bbox_inches="tight")
plt.show()

print("Bimodality here (mass at 0 and 1) means the label is close to deterministic per query;")
print("mass in the middle means k=8 was doing real work and a single roll would have been noisy.")

### Post-hoc confidence features (router 5)

Computed by **teacher-forcing the cached generation back through the weak model** and reading off
the per-token distribution, rather than by capturing `scores` during sampling. Two reasons: the
sampling-time scores for `k = 8` at a 152k vocabulary are tens of GB of logits, and teacher
forcing makes the feature recomputable from the JSON cache without ever re-sampling.

Three features per query: mean token entropy, mean max-probability, and min max-probability over
the generated span. Entropy is taken at temperature 1.0 on the model's own distribution over the
tokens it actually produced -- not at the sampling temperature -- so the feature measures the
model's belief rather than the decoder's setting.

**This router is not cost-free.** It needs the completion to exist, so a query it declines to
escalate has still cost a full weak-model generation, and a query it escalates has cost a weak
generation *plus* a strong one. Section 6 charges it on exactly those terms.

In [ ]:
@torch.no_grad()
def posthoc_confidence(
    model, tok, prompts: Sequence[str], completions: Sequence[str], batch_size: int
) -> np.ndarray:
    """Compute token-level confidence features by teacher-forcing completions.

    Args:
        model: Causal LM in eval mode.
        tok: Its tokenizer.
        prompts: Prompt strings, one per query.
        completions: The completion to score for each query (same length).
        batch_size: Progress-reporting chunk size. Sequences are scored one at a
            time regardless, because a single forward pass already produces a
            `seq_len * vocab` logits tensor at a ~152k vocabulary; this argument
            controls only how often memory is released and progress printed.

    Returns:
        Array of shape (n, 3) with columns
        `[mean_entropy, mean_max_prob, min_max_prob]` over the generated span,
        float32. Rows whose completion tokenises to nothing are filled with NaN.

    Raises:
        ValueError: If `prompts` and `completions` differ in length.
    """
    if len(prompts) != len(completions):
        raise ValueError("prompts and completions must have the same length")

    device = next(model.parameters()).device
    feats = np.full((len(prompts), 3), np.nan, dtype=np.float32)

    for start in range(0, len(prompts), batch_size):
        idxs = list(range(start, min(start + batch_size, len(prompts))))
        for i in idxs:
            p_ids = tok(prompts[i], return_tensors="pt")["input_ids"][0]
            c_ids = tok(completions[i], return_tensors="pt", add_special_tokens=False)["input_ids"][0]
            if c_ids.numel() == 0:
                continue
            ids = torch.cat([p_ids, c_ids]).unsqueeze(0).to(device)
            logits = model(input_ids=ids).logits.float()
            # Position t predicts token t+1, so the distributions that generated
            # the completion start at index len(prompt)-1.
            span = logits[0, p_ids.numel() - 1 : -1, :]
            logp = torch.log_softmax(span, dim=-1)
            p = logp.exp()
            ent = -(p * logp).sum(dim=-1)
            mx = p.max(dim=-1).values
            feats[i] = [float(ent.mean()), float(mx.mean()), float(mx.min())]
            del logits, span, logp, p
        if start % (batch_size * 20) == 0:
            print(f"  posthoc {min(start + batch_size, len(prompts))}/{len(prompts)}", flush=True)
        free_memory()
    return feats


def ensure_posthoc(dataset: str, model, tok) -> np.ndarray:
    """Return cached post-hoc confidence features, computing them if absent.

    Args:
        dataset: Dataset name.
        model: Loaded weak model, or None if the cache is expected to hit.
        tok: Its tokenizer, or None if the cache is expected to hit.

    Returns:
        Array (n, 3) of confidence features, float32.

    Raises:
        RuntimeError: If the cache misses and `model` is None.
    """
    path = cache_npz_path(
        "posthoc",
        model=CONFIG["model_name"],
        dataset=dataset,
        n=CONFIG["n_examples"][dataset],
        sample_index=CONFIG["posthoc_sample_index"],
        k=CONFIG["k_samples"],
        temperature=CONFIG["temperature"],
        seed=CONFIG["seed"],
    )
    if path.exists():
        print(f"[cache hit ] posthoc/{dataset}  {path.name}")
        return np.load(path)["feats"].astype(np.float32)
    if model is None:
        raise RuntimeError(f"posthoc cache miss for {dataset} at {path} and no model supplied")

    print(f"[cache miss] posthoc/{dataset}  computing -> {path.name}")
    j = CONFIG["posthoc_sample_index"]
    prompts = [r["prompt"] for r in RECORDS[dataset]]
    comps = [g[j] for g in WEAK_GENS[dataset]["generations"]]
    feats = posthoc_confidence(model, tok, prompts, comps, CONFIG["posthoc_batch_size"])
    np.savez_compressed(path, feats=feats)
    return feats


POSTHOC: Dict[str, np.ndarray] = {
    ds: ensure_posthoc(ds, WEAK_MODEL, WEAK_TOK) for ds in CONFIG["datasets"]
}
for _ds in CONFIG["datasets"]:
    _n_nan = int(np.isnan(POSTHOC[_ds]).any(axis=1).sum())
    print(f"{_ds:6s}  posthoc shape={POSTHOC[_ds].shape}  rows_with_nan={_n_nan}")

### Strong-model reference labels

E3 and E5 need to know, per query, whether the *strong* model would have got it right -- otherwise
the "accuracy after escalation" axis is an assumption rather than a measurement.

The strong model sees **the identical prompt**, including the same few-shot prefix. That is
deliberate: routing must be evaluated with the query held fixed, so any accuracy difference is
attributable to the model and not to a prompt rewrite. It does mean an instruct-tuned strong model
is being used slightly out of its native chat format, which understates it. Noted in Limitations.

> **Contestable choice: `strong_k_samples = 1`, greedy.** The strong model's correctness enters
> only as the post-escalation outcome, where the quantity of interest is its expected accuracy,
> not its pass-rate distribution -- and a greedy roll costs 1/8th of what the weak labels cost.
> The cost is that strong correctness carries single-roll noise the weak labels do not. Raise
> `strong_k_samples` if the E3 frontier looks unstable.

Set `CONFIG["run_strong_model"] = False` to skip this on a small GPU; Section 6 then falls back to
a **stated constant** strong accuracy and says so loudly on every plot.

> 🌐 **Model download + heavy compute.** Downloads `Qwen/Qwen2.5-7B-Instruct` (~15 GB) and
> greedily decodes one completion per query per dataset. Skipped entirely when
> `CONFIG["run_strong_model"]` is False.

In [ ]:
STRONG_CORRECT: Dict[str, Optional[np.ndarray]] = {ds: None for ds in CONFIG["datasets"]}
STRONG_OUT_CHARS: Dict[str, Optional[np.ndarray]] = {ds: None for ds in CONFIG["datasets"]}

if CONFIG["run_strong_model"]:
    _all_cached = all(generation_cache_path("strong", ds).exists() for ds in CONFIG["datasets"])
    strong_model, strong_tok = (None, None)
    if not _all_cached:
        strong_model, strong_tok = load_lm(
            CONFIG["strong_model_name"], DEVICE, CONFIG["strong_load_in_4bit"]
        )
        print(f"loaded strong model {CONFIG['strong_model_name']}")

    for _ds in CONFIG["datasets"]:
        _payload = ensure_generations("strong", _ds, strong_model, strong_tok)
        _refs = {r["qid"]: r["reference"] for r in RECORDS[_ds]}
        _ok, _lens = [], []
        for _qid, _gens in zip(_payload["qids"], _payload["generations"]):
            _ok.append(float(np.mean([grade(_ds, g, _refs[_qid]) for g in _gens])))
            _lens.append(float(np.median([len(g) for g in _gens])))
        STRONG_CORRECT[_ds] = np.asarray(_ok, dtype=float)
        STRONG_OUT_CHARS[_ds] = np.asarray(_lens, dtype=float)
        print(f"{_ds:6s}  strong accuracy={STRONG_CORRECT[_ds].mean():.4f}  "
              f"weak accuracy={LABELS[_ds]['pass_rate'].mean():.4f}")

    if strong_model is not None:
        del strong_model, strong_tok
        free_memory()
else:
    print("run_strong_model=False -- E3/E5 will use the stated fallback constant from Section 6.")

## Section 3 - Activation extraction

One forward pass per prompt, **no generation**, capturing the hidden state at the **final prompt
token** from **every layer** (including the embedding output, index 0), in **float32**.

Result per dataset: an array of shape `(n_queries, n_layers + 1, hidden_size)`.

> **Contestable choice: pooling.** Last-token pooling is used because it is the only position
> whose representation the model has actually committed to before decoding -- it is literally the
> state the next token is read off. Alternatives: mean-pooling over prompt tokens, which is
> smoother but mixes in the constant few-shot prefix and so leaks prompt-length structure directly
> into the feature; or max-pooling, which is unstable across layers with different activation
> scales. Last-token also keeps the feature exactly what a deployed router would have available at
> zero marginal cost, since the prefill already computes it.
>
> **Contestable choice: float32 storage.** The model runs in bf16 on CUDA, and the hidden states
> are upcast to float32 before storage. This costs 2x disk and buys the linear algebra downstream
> a stable scale; bf16 has ~8 bits of mantissa, and standardising then inverting a covariance in
> that precision is asking for trouble.

Cached as `.npz` keyed by `(model, dataset, n_examples, ...)` so this runs once.

In [ ]:
@torch.no_grad()
def extract_activations(
    model, tok, prompts: Sequence[str], batch_size: int, max_prompt_tokens: int, verbose: bool = True
) -> Tuple[np.ndarray, np.ndarray]:
    """Capture the final-prompt-token hidden state at every layer, before generation.

    With left padding the final prompt token is always at position -1, so no
    per-row index arithmetic is needed and no pad token can be selected.

    Args:
        model: Causal LM in eval mode.
        tok: Its tokenizer, with `padding_side == "left"`.
        prompts: Prompt strings.
        batch_size: Prompts per forward pass.
        max_prompt_tokens: Left-truncation limit; prompts longer than this are
            truncated from the left, preserving the query at the end.
        verbose: Print progress.

    Returns:
        `(acts, lengths)` where `acts` has shape
        `(n, n_layers + 1, hidden_size)` in float32 and `lengths` is the
        untruncated prompt length in tokens, int32.

    Raises:
        ValueError: If `prompts` is empty.
    """
    if len(prompts) == 0:
        raise ValueError("prompts is empty")

    device = next(model.parameters()).device
    tok.truncation_side = "left"
    lengths = np.asarray(
        [len(tok(p)["input_ids"]) for p in prompts], dtype=np.int32
    )

    acts: Optional[np.ndarray] = None
    t0 = time.time()
    for start in range(0, len(prompts), batch_size):
        chunk = list(prompts[start : start + batch_size])
        enc = tok(
            chunk, return_tensors="pt", padding=True,
            truncation=True, max_length=max_prompt_tokens,
        ).to(device)
        out = model(**enc, output_hidden_states=True, use_cache=False)
        # hidden_states: tuple of (n_layers + 1) tensors, each (B, T, H). Slice the
        # final position out of each layer BEFORE stacking -- stacking first would
        # materialise a (B, n_layers + 1, T, H) tensor for no reason.
        hs = torch.stack([h[:, -1, :] for h in out.hidden_states], dim=1).float().cpu().numpy()
        if acts is None:
            acts = np.zeros((len(prompts), hs.shape[1], hs.shape[2]), dtype=np.float32)
        acts[start : start + len(chunk)] = hs
        del out, hs
        if verbose:
            done = min(start + batch_size, len(prompts))
            print(f"  activations {done}/{len(prompts)}  "
                  f"({done / max(time.time() - t0, 1e-9):.1f} prompt/s)", flush=True)
        free_memory()

    assert acts is not None
    return acts, lengths


def ensure_activations(dataset: str, model, tok) -> Tuple[np.ndarray, np.ndarray]:
    """Return cached activations and prompt lengths, extracting them if absent.

    Args:
        dataset: Dataset name.
        model: Loaded weak model, or None if the cache is expected to hit.
        tok: Its tokenizer, or None if the cache is expected to hit.

    Returns:
        `(acts, lengths)` as returned by `extract_activations`.

    Raises:
        RuntimeError: If the cache misses and `model` is None.
    """
    path = cache_npz_path(
        "acts",
        model=CONFIG["model_name"],
        dataset=dataset,
        n=CONFIG["n_examples"][dataset],
        n_fewshot=CONFIG["n_fewshot"][dataset],
        max_prompt_tokens=CONFIG["act_max_prompt_tokens"],
        seed=CONFIG["seed"],
    )
    if path.exists():
        print(f"[cache hit ] acts/{dataset}  {path.name}")
        z = np.load(path)
        return z["acts"].astype(np.float32), z["lengths"].astype(np.int32)
    if model is None:
        raise RuntimeError(f"activation cache miss for {dataset} at {path} and no model supplied")

    print(f"[cache miss] acts/{dataset}  extracting -> {path.name}")
    prompts = [r["prompt"] for r in RECORDS[dataset]]
    acts, lengths = extract_activations(
        model, tok, prompts, CONFIG["act_batch_size"], CONFIG["act_max_prompt_tokens"]
    )
    np.savez_compressed(path, acts=acts, lengths=lengths)
    return acts, lengths

In [ ]:
ACTS: Dict[str, np.ndarray] = {}
PROMPT_LEN: Dict[str, np.ndarray] = {}

for _ds in CONFIG["datasets"]:
    ACTS[_ds], PROMPT_LEN[_ds] = ensure_activations(_ds, WEAK_MODEL, WEAK_TOK)
    _a = ACTS[_ds]
    print(f"{_ds:6s}  acts={_a.shape}  dtype={_a.dtype}  "
          f"{_a.nbytes / 1e6:.1f} MB  prompt_len mean={PROMPT_LEN[_ds].mean():.1f} "
          f"sd={PROMPT_LEN[_ds].std():.1f}")

N_LAYERS = ACTS[CONFIG["datasets"][0]].shape[1] - 1
HIDDEN = ACTS[CONFIG["datasets"][0]].shape[2]
LAYER_GRID = sorted(set(list(range(0, N_LAYERS + 1, CONFIG["layer_grid_stride"])) + [N_LAYERS]))
print(f"\nn_layers (excl. embeddings) = {N_LAYERS}   hidden = {HIDDEN}")
print(f"candidate layers for model selection: {LAYER_GRID}")

### The confound, stated before it is tested

Prompt length is not a nuisance you can wave away: harder GSM8K problems have longer statements,
harder MBPP problems have more unit tests, and the final-token hidden state of a transformer
encodes position. So a probe that "predicts failure from activations" may be reading a positional
signal that a two-parameter logistic regression on token count would have found for free.

That is why router 3 exists, and why Section 5 residualises. The cell below quantifies the
exposure up front -- how strongly prompt length alone relates to the label -- so the E1 numbers
are read with that already on the table.

In [ ]:
rows = []
for ds in CONFIG["datasets"]:
    L = PROMPT_LEN[ds].astype(float)
    y = LABELS[ds]["y"]
    rows.append({
        "dataset": ds,
        "len_mean": L.mean(),
        "len_sd": L.std(),
        "len_min": L.min(),
        "len_max": L.max(),
        "corr(len, pass_rate)": float(np.corrcoef(L, LABELS[ds]["pass_rate"])[0, 1]),
        "AUC(len -> fail)": float(roc_auc_score(y, L)) if len(np.unique(y)) > 1 else np.nan,
    })
length_exposure = pd.DataFrame(rows).set_index("dataset")
display(length_exposure)
print("AUC(len -> fail) is the single-feature, no-fitting version of router 3.")
print("Anything the activation probe achieves must be read against this column.")

In [ ]:
# The weak model's weights are no longer needed: generations, post-hoc features
# and activations are all on disk. The tokenizer is kept -- Section 6 needs it to
# turn cached generations into token counts for the cost model.
del WEAK_MODEL
free_memory()
print("weak model weights released; WEAK_TOK retained for the Section 6 cost model.")
print("Everything below is pure NumPy / scikit-learn over the caches.")

### Query-embedding features (router 4)

A frozen sentence encoder applied to the **raw question text** (not the few-shot prefix, which is
constant and would only add a shared offset). This router knows what the query is *about* but
knows nothing about the weak model, so it isolates how much of the probe's performance is
"this topic is hard in general" versus "this specific model is about to fail".

> 🌐 **Network cell + model download.** Downloads `sentence-transformers/all-MiniLM-L6-v2`
> (~90 MB) on first run. Encoding is CPU-fast; no cache is used because it takes seconds.

In [ ]:
from sentence_transformers import SentenceTransformer


def build_embedding_features(dataset: str, encoder: SentenceTransformer) -> np.ndarray:
    """Encode the raw question text of a dataset with a frozen sentence encoder.

    Args:
        dataset: Dataset name; questions are read from `RECORDS[dataset]`.
        encoder: A loaded SentenceTransformer.

    Returns:
        Array (n, d) of float32 sentence embeddings.

    Raises:
        KeyError: If a record lacks `meta["question"]`.
    """
    texts = [r["meta"]["question"] for r in RECORDS[dataset]]
    return np.asarray(
        encoder.encode(texts, batch_size=64, show_progress_bar=False, convert_to_numpy=True),
        dtype=np.float32,
    )


_encoder = SentenceTransformer(CONFIG["encoder_name"], device=str(DEVICE))
EMBEDS: Dict[str, np.ndarray] = {ds: build_embedding_features(ds, _encoder) for ds in CONFIG["datasets"]}
del _encoder
free_memory()
for _ds in CONFIG["datasets"]:
    print(f"{_ds:6s}  embeddings={EMBEDS[_ds].shape}")

## Section 4 - E1: in-domain probe

**Claim under test (replication):** a linear probe on the weak model's final-prompt-token hidden
state predicts, above chance and above the length baseline, whether the weak model is about to
fail -- with no generation performed.

### Protocol, and what it protects against

- **Outer resampling:** 50 stratified shuffle splits (70/30). Every reported number is the mean
  over those 50 splits with a percentile interval, never a single split. A single split on
  n ~ 1000 with a linear probe on 896 dimensions has a standard error on AUC of roughly 0.02-0.03,
  which is the same size as most of the effects being claimed.
- **Nested selection:** inside each outer split, layer and regularisation strength are chosen by
  5-fold CV **on the training fold only**. The outer test fold never participates in selecting
  anything. Reporting the best layer chosen by looking at test AUC -- the common shortcut -- would
  inflate the headline by selecting over ~13 layers of noise.
- **Preprocessing:** the imputer, the scaler and any PCA are fit inside the pipeline, on the
  training fold, for every split independently.

### What is reported

Two different things, and they should not be confused:

1. **The layer curve** -- AUC per layer per dataset, each layer fit and scored across the same 50
   splits. This is descriptive: it shows *where* in the network the signal lives. It is not a
   headline number, because reading the max off it is selection on the test fold.
2. **The nested-CV AUC** -- one honest number per dataset, where the layer was chosen without
   seeing the test fold. This is the number that goes in the results table, alongside the
   distribution of which layers got selected.

Calibration (Brier score plus a reliability curve) is reported next to every AUC. Routing is a
*thresholded* decision: a probe that ranks well but is badly calibrated cannot have a threshold set
from held-out probabilities, which is exactly the failure mode E5 goes after.

In [ ]:
from sklearn.impute import SimpleImputer


def make_outer_splits(
    y: np.ndarray, n_splits: int, test_size: float, seed: int
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Build stratified train/test resamples used for every CI in this notebook.

    These are repeated stratified holdouts, not a nonparametric bootstrap of a
    fixed test set. The interval they produce therefore covers variation from
    *both* the finite sample and the choice of split, which is the relevant
    uncertainty for "would this probe have worked on other data".

    Args:
        y: Binary label array used for stratification.
        n_splits: Number of resamples.
        test_size: Fraction held out in each resample.
        seed: RNG seed.

    Returns:
        List of `(train_idx, test_idx)` index arrays.

    Raises:
        ValueError: If `y` has fewer than two classes.
    """
    if len(np.unique(y)) < 2:
        raise ValueError("y must contain both classes to stratify")
    sss = StratifiedShuffleSplit(n_splits=n_splits, test_size=test_size, random_state=seed)
    return [(tr, te) for tr, te in sss.split(np.zeros(len(y)), y)]


def make_probe(C: float, pca_components: Optional[int], seed: int) -> Pipeline:
    """Construct the probe pipeline: impute -> standardise -> (PCA) -> logistic regression.

    Every step is fit inside the pipeline, so cloning it per split guarantees no
    preprocessing statistic ever crosses from a test fold into a training fold.

    Args:
        C: Inverse L2 regularisation strength.
        pca_components: Number of components, or None to skip PCA.
        seed: Seed for the solver and PCA.

    Returns:
        An unfitted sklearn `Pipeline`.

    Raises:
        ValueError: If `C <= 0`.
    """
    if C <= 0:
        raise ValueError(f"C must be positive, got {C}")
    steps: List[Tuple[str, Any]] = [
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
    if pca_components is not None:
        steps.append(("pca", PCA(n_components=pca_components, random_state=seed)))
    steps.append((
        "clf",
        LogisticRegression(
            C=C, max_iter=CONFIG["probe_max_iter"], solver="lbfgs", random_state=seed
        ),
    ))
    return Pipeline(steps)


def _safe_auc(y_true: np.ndarray, score: np.ndarray) -> float:
    """ROC AUC that returns NaN instead of raising on a single-class fold.

    Args:
        y_true: Binary labels.
        score: Continuous scores, higher = more likely positive.

    Returns:
        The AUC, or NaN if `y_true` has one class.

    Raises:
        Nothing.
    """
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, score))


def select_C(
    X: np.ndarray, y: np.ndarray, C_grid: Sequence[float], folds: int, seed: int
) -> float:
    """Choose the regularisation strength by inner stratified CV on a training fold.

    Args:
        X: Training-fold features (n, d).
        y: Training-fold binary labels.
        C_grid: Candidate values of `C`.
        folds: Number of inner folds.
        seed: Seed for the fold split and the probe.

    Returns:
        The `C` with the highest mean inner-fold AUC. Ties break toward the
        smallest `C` (strongest regularisation).

    Raises:
        ValueError: If `C_grid` is empty.
    """
    if len(C_grid) == 0:
        raise ValueError("C_grid is empty")
    if len(C_grid) == 1:
        return float(C_grid[0])
    n_min = int(min(np.bincount(y).min(), folds))
    if n_min < 2:
        return float(min(C_grid))
    skf = StratifiedKFold(n_splits=max(2, n_min), shuffle=True, random_state=seed)
    best_C, best_auc = float(min(C_grid)), -np.inf
    for C in C_grid:
        aucs = []
        for tr, va in skf.split(X, y):
            probe = make_probe(C, CONFIG["pca_components"], seed)
            probe.fit(X[tr], y[tr])
            aucs.append(_safe_auc(y[va], probe.predict_proba(X[va])[:, 1]))
        m = float(np.nanmean(aucs))
        if m > best_auc:
            best_C, best_auc = float(C), m
    return best_C


def bootstrap_ci(values: Sequence[float], alpha: float) -> Tuple[float, float, float]:
    """Summarise a set of per-split statistics as mean and percentile interval.

    Args:
        values: Per-split statistics (NaNs are ignored).
        alpha: Two-sided miscoverage, e.g. 0.05 for a 95% interval.

    Returns:
        `(mean, lo, hi)`. All three are NaN if every value is NaN.

    Raises:
        ValueError: If `alpha` is not in (0, 1).
    """
    if not 0.0 < alpha < 1.0:
        raise ValueError(f"alpha must be in (0, 1), got {alpha}")
    v = np.asarray([x for x in values], dtype=float)
    v = v[~np.isnan(v)]
    if v.size == 0:
        return (float("nan"),) * 3
    return (
        float(v.mean()),
        float(np.percentile(v, 100 * alpha / 2)),
        float(np.percentile(v, 100 * (1 - alpha / 2))),
    )

In [ ]:
def evaluate_feature_set(
    X: np.ndarray,
    y: np.ndarray,
    splits: Sequence[Tuple[np.ndarray, np.ndarray]],
    C_grid: Sequence[float],
    inner_folds: int,
    seed: int,
    X_test: Optional[np.ndarray] = None,
    y_test: Optional[np.ndarray] = None,
    test_index_pool: Optional[Sequence[np.ndarray]] = None,
) -> Dict[str, Any]:
    """Fit and score one router on a fixed set of resamples.

    In the in-domain case (`X_test is None`) each split's test fold comes from
    the same array. In the transfer case, `X_test`/`y_test` supply the target
    domain and `test_index_pool` gives, per split, which target rows to score
    on; the source split's *train* indices are still used for fitting, so the
    source-side resampling variation is preserved.

    Args:
        X: Source features (n, d).
        y: Source binary labels.
        splits: `(train_idx, test_idx)` pairs over `X`.
        C_grid: Candidate regularisation strengths for inner CV.
        inner_folds: Inner CV folds used to pick `C`.
        seed: Base seed; split index is added to it.
        X_test: Optional target-domain features.
        y_test: Optional target-domain labels.
        test_index_pool: Optional per-split index arrays into `X_test`.

    Returns:
        Dict with `auc` and `brier` arrays of length `len(splits)`, `C` the
        chosen strengths, `pooled_y` / `pooled_p` concatenating every test
        prediction for calibration plots, and `split_idx` / `split_p` holding
        the per-split evaluation indices and predicted probabilities, which the
        routing simulations in Sections 6 and 8 replay.

    Raises:
        ValueError: If `X_test` is given without `y_test`, or shapes disagree.
    """
    if (X_test is None) != (y_test is None):
        raise ValueError("X_test and y_test must be supplied together")
    if X.shape[0] != y.shape[0]:
        raise ValueError(f"X has {X.shape[0]} rows but y has {y.shape[0]}")

    aucs, briers, chosen = [], [], []
    pooled_y, pooled_p = [], []
    split_idx, split_p = [], []

    for s, (tr, te) in enumerate(splits):
        C = select_C(X[tr], y[tr], C_grid, inner_folds, seed + s)
        probe = make_probe(C, CONFIG["pca_components"], seed + s)
        probe.fit(X[tr], y[tr])

        if X_test is None:
            eval_idx = np.asarray(te)
            Xe, ye = X[te], y[te]
        else:
            eval_idx = np.asarray(
                test_index_pool[s] if test_index_pool is not None else np.arange(len(y_test))
            )
            Xe, ye = X_test[eval_idx], y_test[eval_idx]

        p = probe.predict_proba(Xe)[:, 1]
        aucs.append(_safe_auc(ye, p))
        briers.append(float(brier_score_loss(ye, p)) if len(np.unique(ye)) > 1 else float("nan"))
        chosen.append(C)
        pooled_y.append(ye)
        pooled_p.append(p)
        split_idx.append(eval_idx)
        split_p.append(p)

    return {
        "auc": np.asarray(aucs, dtype=float),
        "brier": np.asarray(briers, dtype=float),
        "C": np.asarray(chosen, dtype=float),
        "pooled_y": np.concatenate(pooled_y),
        "pooled_p": np.concatenate(pooled_p),
        "split_idx": split_idx,
        "split_p": split_p,
    }


def evaluate_random_router(
    y: np.ndarray, splits: Sequence[Tuple[np.ndarray, np.ndarray]], seed: int
) -> Dict[str, Any]:
    """Score a uniform-random router on the same splits, as the lower bound.

    Args:
        y: Binary labels.
        splits: `(train_idx, test_idx)` pairs.
        seed: RNG seed.

    Returns:
        Same dict shape as `evaluate_feature_set`.

    Raises:
        Nothing.
    """
    rng = np.random.default_rng(seed)
    aucs, briers, pooled_y, pooled_p = [], [], [], []
    split_idx, split_p = [], []
    for tr, te in splits:
        p = rng.uniform(size=len(te))
        aucs.append(_safe_auc(y[te], p))
        # A random router's honest probability estimate is the training base rate.
        base = np.full(len(te), float(y[tr].mean()))
        briers.append(float(brier_score_loss(y[te], base)) if len(np.unique(y[te])) > 1 else np.nan)
        pooled_y.append(y[te])
        pooled_p.append(base)
        split_idx.append(np.asarray(te))
        split_p.append(p)
    return {
        "auc": np.asarray(aucs), "brier": np.asarray(briers),
        "C": np.full(len(splits), np.nan),
        "pooled_y": np.concatenate(pooled_y), "pooled_p": np.concatenate(pooled_p),
        "split_idx": split_idx, "split_p": split_p,
    }


def evaluate_oracle_router(
    y: np.ndarray, splits: Sequence[Tuple[np.ndarray, np.ndarray]]
) -> Dict[str, Any]:
    """Score the oracle router, which sees the true label, as the upper bound.

    AUC is 1.0 and Brier is 0.0 by construction. It is included so that the
    tables and frontier plots carry an explicit ceiling rather than an implied one.

    Args:
        y: Binary labels.
        splits: `(train_idx, test_idx)` pairs.

    Returns:
        Same dict shape as `evaluate_feature_set`.

    Raises:
        Nothing.
    """
    pooled_y = np.concatenate([y[te] for _, te in splits])
    return {
        "auc": np.ones(len(splits)), "brier": np.zeros(len(splits)),
        "C": np.full(len(splits), np.nan),
        "pooled_y": pooled_y, "pooled_p": pooled_y.astype(float),
        "split_idx": [np.asarray(te) for _, te in splits],
        "split_p": [y[te].astype(float) for _, te in splits],
    }

In [ ]:
SPLITS: Dict[str, List[Tuple[np.ndarray, np.ndarray]]] = {
    ds: make_outer_splits(
        LABELS[ds]["y"], CONFIG["n_outer_splits"], CONFIG["outer_test_size"], CONFIG["seed"]
    )
    for ds in CONFIG["datasets"]
}
for _ds in CONFIG["datasets"]:
    _tr, _te = SPLITS[_ds][0]
    print(f"{_ds:6s}  {CONFIG['n_outer_splits']} splits, train={len(_tr)} test={len(_te)}, "
          f"test fail-rate={LABELS[_ds]['y'][_te].mean():.3f}")

In [ ]:
def layer_auc_curve(
    acts: np.ndarray,
    y: np.ndarray,
    splits: Sequence[Tuple[np.ndarray, np.ndarray]],
    layers: Sequence[int],
    C_grid: Sequence[float],
    inner_folds: int,
    seed: int,
) -> pd.DataFrame:
    """Score a probe at each candidate layer across the same outer splits.

    Descriptive only: the maximum of this curve is a selection-on-test estimate
    and must not be quoted as the probe's performance. `nested_layer_probe`
    supplies the honest number.

    Args:
        acts: Activations (n, n_layers + 1, hidden).
        y: Binary labels.
        splits: Outer `(train_idx, test_idx)` pairs.
        layers: Candidate layer indices into axis 1 of `acts`.
        C_grid: Candidate regularisation strengths.
        inner_folds: Inner CV folds.
        seed: Base seed.

    Returns:
        DataFrame indexed by layer with columns `auc_mean`, `auc_lo`, `auc_hi`,
        `brier_mean`.

    Raises:
        IndexError: If any layer index is out of range for `acts`.
    """
    rows = []
    for layer in layers:
        if not 0 <= layer < acts.shape[1]:
            raise IndexError(f"layer {layer} out of range for acts with {acts.shape[1]} layers")
        res = evaluate_feature_set(
            acts[:, layer, :], y, splits, C_grid, inner_folds, seed
        )
        m, lo, hi = bootstrap_ci(res["auc"], CONFIG["ci_alpha"])
        rows.append({
            "layer": layer, "auc_mean": m, "auc_lo": lo, "auc_hi": hi,
            "brier_mean": float(np.nanmean(res["brier"])),
        })
        print(f"    layer {layer:3d}  AUC={m:.4f} [{lo:.4f}, {hi:.4f}]", flush=True)
    return pd.DataFrame(rows).set_index("layer")


LAYER_CURVES: Dict[str, pd.DataFrame] = {}
for _ds in CONFIG["datasets"]:
    print(f"[E1] layer curve for {_ds}")
    LAYER_CURVES[_ds] = layer_auc_curve(
        ACTS[_ds], LABELS[_ds]["y"], SPLITS[_ds], LAYER_GRID,
        CONFIG["probe_C_grid"], CONFIG["inner_folds"], CONFIG["seed"],
    )
    display(LAYER_CURVES[_ds])

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.4))
for ds in CONFIG["datasets"]:
    c = LAYER_CURVES[ds]
    ax.plot(c.index, c["auc_mean"], marker="o", label=ds)
    ax.fill_between(c.index, c["auc_lo"], c["auc_hi"], alpha=0.15)
    ax.axhline(length_exposure.loc[ds, "AUC(len -> fail)"], linestyle=":", linewidth=1,
               color=ax.lines[-1].get_color())
ax.axhline(0.5, color="black", linestyle="--", linewidth=1, label="chance")
ax.set_xlabel(f"layer (0 = embedding output, {N_LAYERS} = final)")
ax.set_ylabel("AUC for predicting weak-model failure")
ax.set_title("E1: where in the network the pre-generation failure signal lives\n"
             "(bands = 95% over 50 splits; dotted lines = length-only AUC per dataset)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "e1_layer_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def nested_layer_probe(
    acts: np.ndarray,
    y: np.ndarray,
    splits: Sequence[Tuple[np.ndarray, np.ndarray]],
    layers: Sequence[int],
    C_grid: Sequence[float],
    inner_folds: int,
    seed: int,
    acts_test: Optional[np.ndarray] = None,
    y_test: Optional[np.ndarray] = None,
    test_index_pool: Optional[Sequence[np.ndarray]] = None,
) -> Dict[str, Any]:
    """Fit a probe whose layer AND regularisation are chosen on the training fold only.

    For each outer split: run inner stratified CV over `layers x C_grid` using
    only the training rows, take the argmax, refit on the whole training fold,
    and score once on the held-out fold (or on the target domain, for transfer).

    Args:
        acts: Source activations (n, n_layers + 1, hidden).
        y: Source binary labels.
        splits: Outer `(train_idx, test_idx)` pairs over the source.
        layers: Candidate layer indices.
        C_grid: Candidate regularisation strengths.
        inner_folds: Inner CV folds.
        seed: Base seed; the split index is added to it.
        acts_test: Optional target-domain activations for transfer.
        y_test: Optional target-domain labels for transfer.
        test_index_pool: Optional per-split index arrays into `acts_test`.

    Returns:
        Dict with `auc`, `brier`, `layer` and `C` arrays of length
        `len(splits)`, pooled `pooled_y` / `pooled_p` for calibration, and
        per-split `split_idx` / `split_p` for the routing simulations.

    Raises:
        ValueError: If `acts_test` is supplied without `y_test`.
    """
    if (acts_test is None) != (y_test is None):
        raise ValueError("acts_test and y_test must be supplied together")

    aucs, briers, sel_layers, sel_C = [], [], [], []
    pooled_y, pooled_p = [], []
    split_idx, split_p = [], []

    for s, (tr, te) in enumerate(splits):
        rs = seed + s
        n_min = int(min(np.bincount(y[tr]).min(), inner_folds))
        skf = StratifiedKFold(n_splits=max(2, n_min), shuffle=True, random_state=rs)
        inner_folds_idx = list(skf.split(np.zeros(len(tr)), y[tr]))

        best = (-np.inf, layers[0], float(min(C_grid)))
        for layer in layers:
            Xtr_full = acts[tr][:, layer, :]
            for C in C_grid:
                scores = []
                for itr, iva in inner_folds_idx:
                    probe = make_probe(C, CONFIG["pca_components"], rs)
                    probe.fit(Xtr_full[itr], y[tr][itr])
                    scores.append(
                        _safe_auc(y[tr][iva], probe.predict_proba(Xtr_full[iva])[:, 1])
                    )
                m = float(np.nanmean(scores))
                if m > best[0]:
                    best = (m, layer, float(C))

        _, layer_star, C_star = best
        probe = make_probe(C_star, CONFIG["pca_components"], rs)
        probe.fit(acts[tr][:, layer_star, :], y[tr])

        if acts_test is None:
            eval_idx = np.asarray(te)
            Xe, ye = acts[te][:, layer_star, :], y[te]
        else:
            eval_idx = np.asarray(
                test_index_pool[s] if test_index_pool is not None else np.arange(len(y_test))
            )
            Xe, ye = acts_test[eval_idx][:, layer_star, :], y_test[eval_idx]

        p = probe.predict_proba(Xe)[:, 1]
        aucs.append(_safe_auc(ye, p))
        briers.append(float(brier_score_loss(ye, p)) if len(np.unique(ye)) > 1 else float("nan"))
        sel_layers.append(layer_star)
        sel_C.append(C_star)
        pooled_y.append(ye)
        pooled_p.append(p)
        split_idx.append(eval_idx)
        split_p.append(p)

    return {
        "auc": np.asarray(aucs, dtype=float),
        "brier": np.asarray(briers, dtype=float),
        "layer": np.asarray(sel_layers, dtype=int),
        "C": np.asarray(sel_C, dtype=float),
        "pooled_y": np.concatenate(pooled_y),
        "pooled_p": np.concatenate(pooled_p),
        "split_idx": split_idx,
        "split_p": split_p,
    }

> ⏱ **Slow cell (CPU-bound, no network).** Nested selection over
> `len(LAYER_GRID) x len(probe_C_grid) x inner_folds x n_outer_splits` logistic-regression fits
> per dataset. At the defaults that is on the order of 10k fits per dataset; expect several
> minutes each. Reduce `n_outer_splits` or `layer_grid_stride` to trade CI width for time.

In [ ]:
E1: Dict[str, Dict[str, Any]] = {}
for _ds in CONFIG["datasets"]:
    print(f"[E1] nested probe for {_ds}", flush=True)
    E1[_ds] = {
        "random":  evaluate_random_router(LABELS[_ds]["y"], SPLITS[_ds], CONFIG["seed"]),
        "oracle":  evaluate_oracle_router(LABELS[_ds]["y"], SPLITS[_ds]),
        "length":  evaluate_feature_set(
            PROMPT_LEN[_ds].astype(np.float32).reshape(-1, 1), LABELS[_ds]["y"],
            SPLITS[_ds], CONFIG["probe_C_grid"], CONFIG["inner_folds"], CONFIG["seed"]),
        "embed":   evaluate_feature_set(
            EMBEDS[_ds], LABELS[_ds]["y"], SPLITS[_ds],
            CONFIG["probe_C_grid"], CONFIG["inner_folds"], CONFIG["seed"]),
        "posthoc": evaluate_feature_set(
            POSTHOC[_ds], LABELS[_ds]["y"], SPLITS[_ds],
            CONFIG["probe_C_grid"], CONFIG["inner_folds"], CONFIG["seed"]),
        "probe":   nested_layer_probe(
            ACTS[_ds], LABELS[_ds]["y"], SPLITS[_ds], LAYER_GRID,
            CONFIG["probe_C_grid"], CONFIG["inner_folds"], CONFIG["seed"]),
    }
    print(f"   done {_ds}", flush=True)

In [ ]:
ROUTER_ORDER = ["random", "length", "embed", "posthoc", "probe", "oracle"]
ROUTER_LABEL = {
    "random": "1. Random",
    "length": "3. Prompt length only",
    "embed": "4. Query embedding",
    "posthoc": "5. Post-hoc confidence*",
    "probe": "6. Activation probe",
    "oracle": "2. Oracle",
}


def summarise_routers(results: Dict[str, Dict[str, Any]], alpha: float) -> pd.DataFrame:
    """Turn a dict of router results into a reportable AUC / Brier table.

    Args:
        results: Mapping router name -> dict as returned by `evaluate_feature_set`.
        alpha: Two-sided miscoverage for the percentile interval.

    Returns:
        DataFrame indexed by router with AUC mean/lo/hi, Brier mean/lo/hi, and
        the mean selected layer where a probe recorded one.

    Raises:
        Nothing.
    """
    rows = []
    for name in ROUTER_ORDER:
        if name not in results:
            continue
        r = results[name]
        am, alo, ahi = bootstrap_ci(r["auc"], alpha)
        bm, blo, bhi = bootstrap_ci(r["brier"], alpha)
        rows.append({
            "router": ROUTER_LABEL.get(name, name),
            "AUC": am, "AUC_lo": alo, "AUC_hi": ahi,
            "Brier": bm, "Brier_lo": blo, "Brier_hi": bhi,
            "layer_mean": float(np.mean(r["layer"])) if "layer" in r else np.nan,
            "layer_mode": float(pd.Series(r["layer"]).mode().iloc[0]) if "layer" in r else np.nan,
        })
    return pd.DataFrame(rows).set_index("router")


E1_TABLES: Dict[str, pd.DataFrame] = {}
for _ds in CONFIG["datasets"]:
    print(f"\n===== E1: {_ds} (n={CONFIG['n_examples'][_ds]}, "
          f"fail rate={LABELS[_ds]['y'].mean():.3f}) =====")
    E1_TABLES[_ds] = summarise_routers(E1[_ds], CONFIG["ci_alpha"])
    display(E1_TABLES[_ds])
print("\n* router 5 requires the weak model to generate first; it is not cost-comparable")
print("  with routers 3, 4 and 6. See Section 6.")

In [ ]:
fig, axes = plt.subplots(1, len(CONFIG["datasets"]), figsize=(4.6 * len(CONFIG["datasets"]), 4.0),
                         sharey=True)
axes = np.atleast_1d(axes)
for ax, ds in zip(axes, CONFIG["datasets"]):
    t = E1_TABLES[ds]
    ypos = np.arange(len(t))
    ax.barh(ypos, t["AUC"], color="steelblue")
    ax.errorbar(t["AUC"], ypos,
                xerr=[t["AUC"] - t["AUC_lo"], t["AUC_hi"] - t["AUC"]],
                fmt="none", ecolor="black", capsize=3)
    ax.axvline(0.5, color="black", linestyle="--", linewidth=1)
    ax.set_yticks(ypos)
    ax.set_yticklabels(t.index)
    ax.set_xlim(0.4, 1.02)
    ax.set_xlabel("AUC (fail detection)")
    ax.set_title(ds)
fig.suptitle("E1: in-domain router comparison, identical labels and identical 50 splits", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "e1_router_bars.png", dpi=150, bbox_inches="tight")
plt.show()

### Calibration, reported alongside AUC

AUC only measures *ranking*. A deployed router does not rank; it compares a probability to a fixed
number and escalates. Two probes with identical AUC can behave completely differently once a
threshold is fixed, if one of them puts its probabilities in a different range. The reliability
curves below, and the Brier scores in the tables above, are what make E5 interpretable.

Predictions are pooled across all 50 splits, so each query contributes roughly
`n_splits * test_size` times. That inflates the effective sample size of the reliability curve
without adding independent information -- it is a display aid, not an inference. The Brier scores
in the table are computed per split and then summarised, which is the honest version.

In [ ]:
def plot_reliability(
    ax, y_true: np.ndarray, p: np.ndarray, n_bins: int, label: str
) -> float:
    """Draw a reliability curve on an axis and return the Brier score.

    Args:
        ax: Matplotlib axis to draw on.
        y_true: Binary outcomes.
        p: Predicted probabilities in [0, 1].
        n_bins: Number of quantile bins.
        label: Legend label.

    Returns:
        The Brier score of `(y_true, p)`.

    Raises:
        ValueError: If `y_true` and `p` differ in length.
    """
    if len(y_true) != len(p):
        raise ValueError("y_true and p must have the same length")
    frac_pos, mean_pred = calibration_curve(y_true, p, n_bins=n_bins, strategy="quantile")
    brier = float(brier_score_loss(y_true, p))
    ax.plot(mean_pred, frac_pos, marker="o", label=f"{label} (Brier={brier:.3f})")
    return brier


fig, axes = plt.subplots(1, len(CONFIG["datasets"]), figsize=(4.6 * len(CONFIG["datasets"]), 4.2))
axes = np.atleast_1d(axes)
for ax, ds in zip(axes, CONFIG["datasets"]):
    for name in ["length", "embed", "posthoc", "probe"]:
        plot_reliability(ax, E1[ds][name]["pooled_y"], E1[ds][name]["pooled_p"],
                         CONFIG["calibration_bins"], ROUTER_LABEL[name])
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="perfect")
    ax.set_xlabel("predicted P(fail)")
    ax.set_ylabel("observed failure fraction")
    ax.set_title(ds)
    ax.legend(fontsize=7)
fig.suptitle("E1: reliability curves, pooled over 50 splits", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "e1_calibration.png", dpi=150, bbox_inches="tight")
plt.show()

## Section 5 - E2: confound removal

The worry, stated concretely: a transformer's final-token hidden state is a function of position,
and prompt length correlates with difficulty in all three datasets (Section 3 quantified this). If
the probe is reading a positional encoding of "this prompt is long", then the headline of E1 is a
restatement of the length baseline in 896 dimensions, and none of it transfers, because length
distributions differ wildly across domains.

**Procedure.** For every hidden dimension `d`, fit an OLS regression of `x_d` on
`[1, prompt_length]` **using the training fold only**, then replace `x_d` by its residual on both
folds using those train-fold coefficients. Refit the probe on residuals. Anything that survives is,
by construction, linearly independent of prompt length.

Three numbers go side by side per dataset: **raw AUC**, **residualised AUC**, and the
**length-only baseline**. The interesting quantity is the gap between the second and the third.

> **Contestable choice: linear residualisation.** OLS on `[1, L]` removes only the *linear*
> component of the length relationship. If the dependence is curved -- plausible, since attention
> patterns do not scale linearly in context length -- some length information survives and the
> residualised AUC is an over-estimate of the length-free signal. The alternatives are a spline or
> polynomial basis in `L` (removes more, but starts removing genuine difficulty signal that merely
> correlates with length), or length-stratified matched sampling (cleanest, but throws away most of
> the data at these sample sizes). Linear is the conservative-but-honest middle; the direction of
> its bias is stated rather than hidden.
>
> Note also that residualising against length removes signal that is *genuinely* about difficulty
> whenever difficulty and length are correlated for real reasons. The residualised AUC is therefore
> a **lower bound** on the probe's true non-trivial signal, not an unbiased estimate of it. Both
> bounds are reported for that reason.

In [ ]:
def fit_length_residualiser(
    X_train: np.ndarray, len_train: np.ndarray
) -> Tuple[np.ndarray, np.ndarray]:
    """Fit a per-dimension OLS of features on prompt length, on the training fold only.

    Args:
        X_train: Training features (n_train, d).
        len_train: Training prompt lengths in tokens (n_train,).

    Returns:
        `(slope, intercept)`, each of shape (d,), such that the residual of a
        row is `x - (slope * length + intercept)`.

    Raises:
        ValueError: If row counts disagree or the training lengths are constant
            (in which case the regression is degenerate and residualisation is
            meaningless).
    """
    if X_train.shape[0] != len_train.shape[0]:
        raise ValueError("X_train and len_train must have the same number of rows")
    L = np.asarray(len_train, dtype=np.float64)
    if np.allclose(L.std(), 0.0):
        raise ValueError("prompt lengths are constant on the training fold; cannot residualise")
    ols = LinearRegression()
    ols.fit(L.reshape(-1, 1), X_train.astype(np.float64))
    return ols.coef_.ravel().astype(np.float32), ols.intercept_.astype(np.float32)


def apply_length_residualiser(
    X: np.ndarray, lengths: np.ndarray, slope: np.ndarray, intercept: np.ndarray
) -> np.ndarray:
    """Subtract the fitted length component from every feature dimension.

    Args:
        X: Features (n, d).
        lengths: Prompt lengths (n,).
        slope: Per-dimension slopes (d,) from `fit_length_residualiser`.
        intercept: Per-dimension intercepts (d,).

    Returns:
        Residualised features (n, d), float32.

    Raises:
        ValueError: If the feature width does not match `slope`.
    """
    if X.shape[1] != slope.shape[0]:
        raise ValueError(f"X has width {X.shape[1]} but slope has width {slope.shape[0]}")
    L = np.asarray(lengths, dtype=np.float32).reshape(-1, 1)
    return (X.astype(np.float32) - (L * slope.reshape(1, -1) + intercept.reshape(1, -1)))


def nested_layer_probe_residualised(
    acts: np.ndarray,
    y: np.ndarray,
    lengths: np.ndarray,
    splits: Sequence[Tuple[np.ndarray, np.ndarray]],
    layers: Sequence[int],
    C_grid: Sequence[float],
    inner_folds: int,
    seed: int,
) -> Dict[str, Any]:
    """Run the nested-CV probe on length-residualised activations.

    The residualiser is fit on the outer training fold and applied to both
    folds, so the outer test fold contributes nothing to it. The inner CV then
    runs on already-residualised training rows; the residualiser has therefore
    seen the inner validation rows, which mildly optimises the *selection* of
    layer and `C` but cannot leak into the reported outer-fold score.

    Args:
        acts: Activations (n, n_layers + 1, hidden).
        y: Binary labels.
        lengths: Prompt lengths in tokens (n,).
        splits: Outer `(train_idx, test_idx)` pairs.
        layers: Candidate layer indices.
        C_grid: Candidate regularisation strengths.
        inner_folds: Inner CV folds.
        seed: Base seed.

    Returns:
        Same dict shape as `nested_layer_probe`, plus `resid_len_corr`: the mean
        absolute correlation between the residualised test-fold probe score and
        prompt length, which should be near zero if residualisation worked.

    Raises:
        ValueError: If array lengths disagree.
    """
    if not (acts.shape[0] == y.shape[0] == lengths.shape[0]):
        raise ValueError("acts, y and lengths must have the same number of rows")

    aucs, briers, sel_layers, sel_C, corrs = [], [], [], [], []
    pooled_y, pooled_p, split_idx, split_p = [], [], [], []

    for s, (tr, te) in enumerate(splits):
        rs = seed + s
        n_min = int(min(np.bincount(y[tr]).min(), inner_folds))
        skf = StratifiedKFold(n_splits=max(2, n_min), shuffle=True, random_state=rs)
        inner_idx = list(skf.split(np.zeros(len(tr)), y[tr]))

        best = (-np.inf, layers[0], float(min(C_grid)), None)
        for layer in layers:
            slope, intercept = fit_length_residualiser(acts[tr][:, layer, :], lengths[tr])
            Rtr = apply_length_residualiser(acts[tr][:, layer, :], lengths[tr], slope, intercept)
            for C in C_grid:
                scores = []
                for itr, iva in inner_idx:
                    probe = make_probe(C, CONFIG["pca_components"], rs)
                    probe.fit(Rtr[itr], y[tr][itr])
                    scores.append(_safe_auc(y[tr][iva], probe.predict_proba(Rtr[iva])[:, 1]))
                m = float(np.nanmean(scores))
                if m > best[0]:
                    best = (m, layer, float(C), (slope, intercept))

        _, layer_star, C_star, coef = best
        if coef is None:
            # Every inner fold scored NaN (single-class validation folds). Fall back
            # to the default layer/C rather than crashing on an unfitted residualiser.
            coef = fit_length_residualiser(acts[tr][:, layer_star, :], lengths[tr])
        slope, intercept = coef
        Rtr = apply_length_residualiser(acts[tr][:, layer_star, :], lengths[tr], slope, intercept)
        Rte = apply_length_residualiser(acts[te][:, layer_star, :], lengths[te], slope, intercept)

        probe = make_probe(C_star, CONFIG["pca_components"], rs)
        probe.fit(Rtr, y[tr])
        p = probe.predict_proba(Rte)[:, 1]

        aucs.append(_safe_auc(y[te], p))
        briers.append(float(brier_score_loss(y[te], p)) if len(np.unique(y[te])) > 1 else np.nan)
        sel_layers.append(layer_star)
        sel_C.append(C_star)
        corrs.append(abs(float(np.corrcoef(p, lengths[te].astype(float))[0, 1])))
        pooled_y.append(y[te]); pooled_p.append(p)
        split_idx.append(np.asarray(te)); split_p.append(p)

    return {
        "auc": np.asarray(aucs, dtype=float),
        "brier": np.asarray(briers, dtype=float),
        "layer": np.asarray(sel_layers, dtype=int),
        "C": np.asarray(sel_C, dtype=float),
        "resid_len_corr": np.asarray(corrs, dtype=float),
        "pooled_y": np.concatenate(pooled_y),
        "pooled_p": np.concatenate(pooled_p),
        "split_idx": split_idx,
        "split_p": split_p,
    }

> ⏱ **Slow cell.** Same nested grid as E1 plus an 896-target OLS fit per (split, layer). Budget
> roughly the same wall-clock as the E1 nested cell.

In [ ]:
E2: Dict[str, Dict[str, Any]] = {}
for _ds in CONFIG["datasets"]:
    print(f"[E2] residualised nested probe for {_ds}", flush=True)
    E2[_ds] = nested_layer_probe_residualised(
        ACTS[_ds], LABELS[_ds]["y"], PROMPT_LEN[_ds], SPLITS[_ds], LAYER_GRID,
        CONFIG["probe_C_grid"], CONFIG["inner_folds"], CONFIG["seed"],
    )
    print(f"   done {_ds}", flush=True)

In [ ]:
rows = []
for ds in CONFIG["datasets"]:
    raw_m, raw_lo, raw_hi = bootstrap_ci(E1[ds]["probe"]["auc"], CONFIG["ci_alpha"])
    res_m, res_lo, res_hi = bootstrap_ci(E2[ds]["auc"], CONFIG["ci_alpha"])
    len_m, len_lo, len_hi = bootstrap_ci(E1[ds]["length"]["auc"], CONFIG["ci_alpha"])
    # Paired over splits: the same 50 resamples underlie both probes.
    d_m, d_lo, d_hi = bootstrap_ci(E2[ds]["auc"] - E1[ds]["probe"]["auc"], CONFIG["ci_alpha"])
    g_m, g_lo, g_hi = bootstrap_ci(E2[ds]["auc"] - E1[ds]["length"]["auc"], CONFIG["ci_alpha"])
    rows.append({
        "dataset": ds,
        "probe_raw_AUC": raw_m, "raw_lo": raw_lo, "raw_hi": raw_hi,
        "probe_resid_AUC": res_m, "resid_lo": res_lo, "resid_hi": res_hi,
        "length_only_AUC": len_m, "len_lo": len_lo, "len_hi": len_hi,
        "d(resid - raw)": d_m, "d_lo": d_lo, "d_hi": d_hi,
        "d(resid - length)": g_m, "g_lo": g_lo, "g_hi": g_hi,
        "|corr(score, len)| after": float(E2[ds]["resid_len_corr"].mean()),
    })
E2_TABLE = pd.DataFrame(rows).set_index("dataset")
display(E2_TABLE)

print("Read the two paired-difference columns, not the levels:")
print("  d(resid - raw)    : how much of the raw probe was length. Near 0 => the confound was not")
print("                      driving it. Strongly negative => much of E1 was length.")
print("  d(resid - length) : the length-free advantage over the confound baseline. This is the")
print("                      column that has to be positive with a CI clear of 0 for the probe")
print("                      to be doing anything a token counter could not.")

In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 4.2))
w = 0.26
x = np.arange(len(CONFIG["datasets"]))
series = [
    ("length_only_AUC", "len_lo", "len_hi", "3. prompt length only", "#b0b0b0"),
    ("probe_raw_AUC", "raw_lo", "raw_hi", "6. probe, raw", "#4c72b0"),
    ("probe_resid_AUC", "resid_lo", "resid_hi", "6. probe, length-residualised", "#dd8452"),
]
for i, (col, lo, hi, lab, colr) in enumerate(series):
    v = E2_TABLE[col].values
    ax.bar(x + (i - 1) * w, v, width=w, label=lab, color=colr)
    ax.errorbar(x + (i - 1) * w, v,
                yerr=[v - E2_TABLE[lo].values, E2_TABLE[hi].values - v],
                fmt="none", ecolor="black", capsize=3)
ax.axhline(0.5, color="black", linestyle="--", linewidth=1)
ax.set_xticks(x); ax.set_xticklabels(CONFIG["datasets"])
ax.set_ylabel("AUC (fail detection)")
ax.set_ylim(0.4, 1.0)
ax.set_title("E2: does the pre-generation signal survive removing prompt length?\n"
             "(bars = mean over 50 splits, whiskers = 95%)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "e2_residualisation.png", dpi=150, bbox_inches="tight")
plt.show()

## Section 6 - E3: the cost-quality frontier

### The cost model, stated plainly

Cost per query is approximated as

```
cost = price_in * (prompt tokens) + price_out * (median output tokens for that model+dataset)
```

**The output term is a median, not the realised length.** The true cost depends on how long the
answer turns out to be, which is not known before routing and varies by a factor of several across
queries. Substituting the dataset-level median makes the cost axis *smooth and comparable*, not
*exact*. Anywhere a cost figure appears below, read it as "relative cost under a median-length
assumption", never as a billing estimate. The prices in `CONFIG["price_per_mtok"]` are placeholders
whose only meaningful content is the **ratio** between weak and strong.

### Who pays for what

This is the part that separates the pre-generation routers from the post-hoc one:

- **Routers 1, 3, 4, 6 (pre-generation).** The decision is made from the prompt alone. An escalated
  query pays *only* the strong model. Escalating at fraction `f` costs
  `(1 - f) * cost_weak + f * cost_strong`.
- **Router 5 (post-hoc confidence).** The decision needs the weak model's answer, so it has already
  been paid for. Escalating at fraction `f` costs `cost_weak + f * cost_strong`. At `f = 0` it is
  as cheap as the weak model; at `f = 1` it is strictly more expensive than never using the weak
  model at all.

### Accuracy

Per-query accuracy is the **expected** accuracy: the weak model's pass rate over its `k` samples,
and the strong model's pass rate over its `k_strong` samples. Using expectations rather than a
single sampled outcome removes a layer of Monte-Carlo noise from the frontier that has nothing to do
with the router.

### The two reference curves

- **Oracle** escalates in descending order of `strong_accuracy - weak_accuracy`. That is the
  accuracy-optimal order, and it is strictly better than "escalate everything the weak model gets
  wrong", because it declines to spend on queries the strong model also fails.
- **Random** escalates a uniformly random subset, which traces a straight line between the
  weak-only and strong-only accuracies.

"Fraction of oracle gain captured" at escalation `f` is
`(acc_router(f) - acc_random(f)) / (acc_oracle(f) - acc_random(f))`, computed per split and then
summarised, so the CI carries through.

In [ ]:
def median_output_tokens(dataset: str, role: str, tok) -> float:
    """Median token count of the cached generations for one (dataset, role).

    This single number stands in for every query's output length in the cost
    model. See the section note on why that is an approximation.

    Args:
        dataset: Dataset name.
        role: "weak" or "strong".
        tok: A tokenizer used to count tokens.

    Returns:
        The median number of generated tokens, as a float.

    Raises:
        FileNotFoundError: If the generation cache for that pair is missing.
    """
    path = generation_cache_path(role, dataset)
    if not path.exists():
        raise FileNotFoundError(f"no generation cache at {path}")
    payload = read_json(path)
    j = min(CONFIG["posthoc_sample_index"], len(payload["generations"][0]) - 1)
    counts = [
        len(tok(g[j], add_special_tokens=False)["input_ids"]) for g in payload["generations"]
    ]
    return float(np.median(counts))


def per_query_costs(dataset: str, tok) -> Dict[str, np.ndarray]:
    """Compute per-query weak and strong costs under the median-output-length model.

    Args:
        dataset: Dataset name.
        tok: Tokenizer used for prompt and output token counts.

    Returns:
        Dict with `weak` and `strong` arrays of per-query cost in the same
        (arbitrary) currency units, plus scalars `weak_out_tok` and
        `strong_out_tok` recording the medians actually used.

    Raises:
        FileNotFoundError: If a required generation cache is missing.
    """
    prompt_tok = PROMPT_LEN[dataset].astype(float)
    pw = CONFIG["price_per_mtok"]["weak"]
    ps = CONFIG["price_per_mtok"]["strong"]
    w_out = median_output_tokens(dataset, "weak", tok)
    s_out = w_out
    if CONFIG["run_strong_model"] and generation_cache_path("strong", dataset).exists():
        s_out = median_output_tokens(dataset, "strong", tok)
    weak = (pw["in"] * prompt_tok + pw["out"] * w_out) / 1e6
    strong = (ps["in"] * prompt_tok + ps["out"] * s_out) / 1e6
    return {"weak": weak, "strong": strong, "weak_out_tok": w_out, "strong_out_tok": s_out}


COSTS: Dict[str, Dict[str, np.ndarray]] = {}
for _ds in CONFIG["datasets"]:
    COSTS[_ds] = per_query_costs(_ds, WEAK_TOK)
    print(f"{_ds:6s}  median out tokens: weak={COSTS[_ds]['weak_out_tok']:.0f} "
          f"strong={COSTS[_ds]['strong_out_tok']:.0f}   "
          f"mean cost ratio strong/weak = "
          f"{COSTS[_ds]['strong'].mean() / COSTS[_ds]['weak'].mean():.2f}x")
print("\nCost axis is RELATIVE and assumes median output length. It is not a billing estimate.")

In [ ]:
FALLBACK_STRONG_ACC = 1.0  # used only when run_strong_model is False; stated on every plot


def strong_accuracy(dataset: str) -> Tuple[np.ndarray, bool]:
    """Return per-query strong-model accuracy, or a flagged constant fallback.

    Args:
        dataset: Dataset name.

    Returns:
        `(acc, measured)` where `acc` is a per-query array and `measured` is
        False when the array is the `FALLBACK_STRONG_ACC` constant rather than
        an actual measurement.

    Raises:
        Nothing.
    """
    if STRONG_CORRECT.get(dataset) is not None:
        return STRONG_CORRECT[dataset], True
    n = len(RECORDS[dataset])
    return np.full(n, FALLBACK_STRONG_ACC, dtype=float), False


def routed_accuracy_curve(
    weak_acc: np.ndarray,
    strong_acc: np.ndarray,
    score: np.ndarray,
    fractions: np.ndarray,
    rng: Optional[np.random.Generator] = None,
) -> np.ndarray:
    """Accuracy as a function of escalation fraction for one ranking of queries.

    Args:
        weak_acc: Per-query expected accuracy of the weak model, in [0, 1].
        strong_acc: Per-query expected accuracy of the strong model, in [0, 1].
        score: Router score; higher means escalate sooner. Ties are broken
            randomly when `rng` is given, otherwise by index.
        fractions: Escalation fractions in [0, 1] to evaluate.
        rng: Optional generator used only for tie-breaking.

    Returns:
        Array of accuracies, same length as `fractions`.

    Raises:
        ValueError: If the per-query arrays disagree in length.
    """
    n = len(weak_acc)
    if not (len(strong_acc) == len(score) == n):
        raise ValueError("weak_acc, strong_acc and score must have equal length")
    jitter = rng.uniform(0, 1e-9, size=n) if rng is not None else np.zeros(n)
    order = np.argsort(-(score.astype(float) + jitter), kind="stable")
    out = np.empty(len(fractions), dtype=float)
    for i, f in enumerate(fractions):
        n_esc = int(round(f * n))
        esc = np.zeros(n, dtype=bool)
        esc[order[:n_esc]] = True
        out[i] = float(np.where(esc, strong_acc, weak_acc).mean())
    return out


def routed_cost_curve(
    cost_weak: np.ndarray,
    cost_strong: np.ndarray,
    score: np.ndarray,
    fractions: np.ndarray,
    pays_weak_always: bool,
) -> np.ndarray:
    """Mean per-query cost as a function of escalation fraction.

    Args:
        cost_weak: Per-query weak-model cost.
        cost_strong: Per-query strong-model cost.
        score: Router score; higher means escalate sooner.
        fractions: Escalation fractions in [0, 1].
        pays_weak_always: True for post-hoc routers, which must generate with the
            weak model before they can decide, so the weak cost is incurred on
            every query including escalated ones.

    Returns:
        Array of mean costs, same length as `fractions`.

    Raises:
        ValueError: If the per-query arrays disagree in length.
    """
    n = len(cost_weak)
    if not (len(cost_strong) == len(score) == n):
        raise ValueError("cost arrays and score must have equal length")
    order = np.argsort(-score.astype(float), kind="stable")
    out = np.empty(len(fractions), dtype=float)
    for i, f in enumerate(fractions):
        esc = np.zeros(n, dtype=bool)
        esc[order[: int(round(f * n))]] = True
        if pays_weak_always:
            c = cost_weak + np.where(esc, cost_strong, 0.0)
        else:
            c = np.where(esc, cost_strong, cost_weak)
        out[i] = float(c.mean())
    return out

In [ ]:
PAYS_WEAK_ALWAYS = {
    "random": False, "length": False, "embed": False,
    "posthoc": True, "probe": False, "oracle": False,
}


def build_frontier(dataset: str, results: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    """Build accuracy/cost-versus-escalation curves for every router on one dataset.

    Each router's curve is computed independently on each of the 50 outer test
    folds using that fold's held-out predictions, then summarised across folds,
    so the frontier inherits the same resampling-based CIs as the AUC tables.

    Args:
        dataset: Dataset name.
        results: Mapping router name -> result dict carrying `split_idx` and
            `split_p` (as produced in Sections 4-5).

    Returns:
        Dict with `fractions`, `acc` (router -> array (n_splits, n_fractions)),
        `cost` (same shape), `oracle_acc`, `random_acc`, `oracle_cost`,
        `random_cost`, `weak_only`, `strong_only`, and `strong_measured`.

    Raises:
        KeyError: If a router result lacks per-split predictions.
    """
    weak_acc = LABELS[dataset]["pass_rate"]
    s_acc, measured = strong_accuracy(dataset)
    cw, cs = COSTS[dataset]["weak"], COSTS[dataset]["strong"]
    fractions = np.linspace(0.0, 1.0, CONFIG["frontier_grid"])
    rng = np.random.default_rng(CONFIG["seed"])

    acc: Dict[str, np.ndarray] = {}
    cost: Dict[str, np.ndarray] = {}
    for name, res in results.items():
        if "split_idx" not in res:
            raise KeyError(f"router {name!r} has no per-split predictions")
        a_rows, c_rows = [], []
        for idx, p in zip(res["split_idx"], res["split_p"]):
            a_rows.append(routed_accuracy_curve(weak_acc[idx], s_acc[idx], p, fractions, rng))
            c_rows.append(routed_cost_curve(cw[idx], cs[idx], p, fractions,
                                            PAYS_WEAK_ALWAYS.get(name, False)))
        acc[name] = np.vstack(a_rows)
        cost[name] = np.vstack(c_rows)

    # Reference curves, evaluated on the same test folds.
    ref_idx = results["probe"]["split_idx"]
    oracle_gain = s_acc - weak_acc
    o_rows, oc_rows, r_rows, rc_rows = [], [], [], []
    for s, idx in enumerate(ref_idx):
        o_rows.append(routed_accuracy_curve(weak_acc[idx], s_acc[idx], oracle_gain[idx], fractions))
        oc_rows.append(routed_cost_curve(cw[idx], cs[idx], oracle_gain[idx], fractions, False))
        rnd = np.random.default_rng(CONFIG["seed"] + 10_000 + s).uniform(size=len(idx))
        r_rows.append(routed_accuracy_curve(weak_acc[idx], s_acc[idx], rnd, fractions))
        rc_rows.append(routed_cost_curve(cw[idx], cs[idx], rnd, fractions, False))

    return {
        "fractions": fractions,
        "acc": acc, "cost": cost,
        "oracle_acc": np.vstack(o_rows), "oracle_cost": np.vstack(oc_rows),
        "random_acc": np.vstack(r_rows), "random_cost": np.vstack(rc_rows),
        "weak_only": float(weak_acc.mean()),
        "strong_only": float(s_acc.mean()),
        "strong_measured": measured,
    }


FRONTIERS: Dict[str, Dict[str, Any]] = {}
for _ds in CONFIG["datasets"]:
    FRONTIERS[_ds] = build_frontier(_ds, E1[_ds])
    _f = FRONTIERS[_ds]
    _tag = "measured" if _f["strong_measured"] else f"FALLBACK CONSTANT {FALLBACK_STRONG_ACC}"
    print(f"{_ds:6s}  weak-only acc={_f['weak_only']:.4f}  "
          f"strong-only acc={_f['strong_only']:.4f} ({_tag})")

In [ ]:
def oracle_gain_captured(frontier: Dict[str, Any], router: str, f: float) -> np.ndarray:
    """Fraction of the oracle's accuracy gain a router captures at one escalation rate.

    Args:
        frontier: Output of `build_frontier`.
        router: Router key present in `frontier["acc"]`.
        f: Escalation fraction; snapped to the nearest grid point.

    Returns:
        Per-split array of captured fractions. Splits where the oracle and the
        random baseline coincide (no headroom) yield NaN.

    Raises:
        KeyError: If `router` is not in the frontier.
    """
    i = int(np.argmin(np.abs(frontier["fractions"] - f)))
    a = frontier["acc"][router][:, i]
    o = frontier["oracle_acc"][:, i]
    r = frontier["random_acc"][:, i]
    denom = o - r
    out = np.where(np.abs(denom) > 1e-12, (a - r) / np.where(denom == 0, 1, denom), np.nan)
    return out


rows = []
for ds in CONFIG["datasets"]:
    fr = FRONTIERS[ds]
    for f in CONFIG["escalation_report_points"]:
        i = int(np.argmin(np.abs(fr["fractions"] - f)))
        for router in ["length", "embed", "posthoc", "probe"]:
            g_m, g_lo, g_hi = bootstrap_ci(oracle_gain_captured(fr, router, f), CONFIG["ci_alpha"])
            a_m, a_lo, a_hi = bootstrap_ci(fr["acc"][router][:, i], CONFIG["ci_alpha"])
            rows.append({
                "dataset": ds, "escalation": f, "router": ROUTER_LABEL[router],
                "accuracy": a_m, "acc_lo": a_lo, "acc_hi": a_hi,
                "oracle_acc": float(fr["oracle_acc"][:, i].mean()),
                "random_acc": float(fr["random_acc"][:, i].mean()),
                "frac_oracle_gain": g_m, "gain_lo": g_lo, "gain_hi": g_hi,
                "mean_cost": float(fr["cost"][router][:, i].mean()),
            })
E3_TABLE = pd.DataFrame(rows).set_index(["dataset", "escalation", "router"]).sort_index()
display(E3_TABLE)
print("frac_oracle_gain is the headline: 0 = no better than escalating at random,")
print("1 = matched a router that already knew the answers.")

In [ ]:
fig, axes = plt.subplots(1, len(CONFIG["datasets"]), figsize=(5.0 * len(CONFIG["datasets"]), 4.4))
axes = np.atleast_1d(axes)
for ax, ds in zip(axes, CONFIG["datasets"]):
    fr = FRONTIERS[ds]
    x = fr["fractions"]
    ax.plot(x, fr["oracle_acc"].mean(0), color="black", linewidth=2, label="2. Oracle")
    ax.plot(x, fr["random_acc"].mean(0), color="grey", linestyle="--", label="1. Random")
    for router, colr in [("length", "#b0b0b0"), ("embed", "#55a868"),
                         ("posthoc", "#c44e52"), ("probe", "#4c72b0")]:
        m = fr["acc"][router].mean(0)
        lo = np.percentile(fr["acc"][router], 100 * CONFIG["ci_alpha"] / 2, axis=0)
        hi = np.percentile(fr["acc"][router], 100 * (1 - CONFIG["ci_alpha"] / 2), axis=0)
        ax.plot(x, m, color=colr, label=ROUTER_LABEL[router])
        ax.fill_between(x, lo, hi, color=colr, alpha=0.15)
    for f in CONFIG["escalation_report_points"]:
        ax.axvline(f, color="black", linewidth=0.5, alpha=0.3)
    ax.set_xlabel("fraction of queries escalated to the strong model")
    ax.set_ylabel("expected accuracy")
    title = ds if fr["strong_measured"] else f"{ds}  [STRONG ACC = FALLBACK {FALLBACK_STRONG_ACC}]"
    ax.set_title(title)
    ax.legend(fontsize=7, loc="lower right")
fig.suptitle("E3: cost-quality frontier (bands = 95% over 50 splits)", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "e3_frontier_escalation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(CONFIG["datasets"]), figsize=(5.0 * len(CONFIG["datasets"]), 4.4))
axes = np.atleast_1d(axes)
for ax, ds in zip(axes, CONFIG["datasets"]):
    fr = FRONTIERS[ds]
    ax.plot(fr["oracle_cost"].mean(0), fr["oracle_acc"].mean(0), color="black",
            linewidth=2, label="2. Oracle")
    ax.plot(fr["random_cost"].mean(0), fr["random_acc"].mean(0), color="grey",
            linestyle="--", label="1. Random")
    for router, colr in [("length", "#b0b0b0"), ("embed", "#55a868"),
                         ("posthoc", "#c44e52"), ("probe", "#4c72b0")]:
        ax.plot(fr["cost"][router].mean(0), fr["acc"][router].mean(0),
                color=colr, label=ROUTER_LABEL[router])
    ax.set_xlabel("mean cost per query (arbitrary units, median-output assumption)")
    ax.set_ylabel("expected accuracy")
    ax.set_title(ds)
    ax.legend(fontsize=7, loc="lower right")
fig.suptitle("E3: the same frontier on a cost axis -- note where router 5 starts,\n"
             "since it has already paid for a weak generation before it can decide", y=1.05)
fig.tight_layout()
fig.savefig(FIG_DIR / "e3_frontier_cost.png", dpi=150, bbox_inches="tight")
plt.show()

## Section 7 - E4: domain shift

**This is the contribution.** E1-E3 establish that the signal exists in-domain, which prior work
already told us. The question a deployed router actually faces is different: it is trained on
whatever traffic was available and then sees something else.

### Design

- **Full 3x3 matrix.** Train on each dataset, test on each dataset, every cell computed with the
  same nested protocol. Layer and `C` are selected on the **source** training fold only -- the
  target domain contributes nothing to model selection, which is the point.
- **The diagonal is the E1 number**, i.e. genuinely held-out within-domain performance, not a
  train-on-all-and-test-on-all cell. Putting the in-domain nested-CV result on the diagonal is what
  makes the off-diagonal drop readable as a transfer gap rather than as an overfitting artefact.
- **Off-diagonal cells** fit on a source training fold and evaluate on a stratified bootstrap
  resample of the entire target dataset, one resample per split, so target-sampling variability is
  in the CI too.
- **Three feature families are transferred, not one.** The probe, the length baseline and the query
  embedding. If the probe's transfer matrix looks like the length baseline's, the transfer is a
  length artefact; if it looks like the embedding's, the probe is carrying topic information rather
  than model-specific self-knowledge. Neither alternative can be ruled out by looking at the probe
  alone, which is why all three matrices are computed.

Set `CONFIG["e4_residualised_transfer"] = True` to add a fourth matrix on length-residualised
activations. It roughly doubles this section's runtime and answers the sharper version of the same
question: does the *length-free* part of the signal transfer?

In [ ]:
CONFIG["e4_residualised_transfer"] = False  # see the note above; roughly 2x runtime


def make_target_resamples(
    y_target: np.ndarray, n_splits: int, seed: int
) -> List[np.ndarray]:
    """Build stratified bootstrap resamples of a target dataset for transfer evaluation.

    Args:
        y_target: Binary labels of the target dataset.
        n_splits: Number of resamples, matched to the source split count.
        seed: RNG seed.

    Returns:
        List of index arrays into the target dataset, each the same length as
        the dataset and drawn with replacement within each class.

    Raises:
        ValueError: If `y_target` has fewer than two classes.
    """
    if len(np.unique(y_target)) < 2:
        raise ValueError("target labels must contain both classes")
    pos = np.flatnonzero(y_target == 1)
    neg = np.flatnonzero(y_target == 0)
    out = []
    for s in range(n_splits):
        rng = np.random.default_rng(seed + 5000 + s)
        out.append(np.sort(np.concatenate([
            rng.choice(pos, size=len(pos), replace=True),
            rng.choice(neg, size=len(neg), replace=True),
        ])))
    return out


def transfer_nested_probe(
    acts_src: np.ndarray,
    y_src: np.ndarray,
    len_src: np.ndarray,
    splits_src: Sequence[Tuple[np.ndarray, np.ndarray]],
    acts_tgt: np.ndarray,
    y_tgt: np.ndarray,
    len_tgt: np.ndarray,
    target_pool: Sequence[np.ndarray],
    layers: Sequence[int],
    C_grid: Sequence[float],
    inner_folds: int,
    seed: int,
    residualise: bool,
) -> Dict[str, Any]:
    """Train a nested-CV probe on a source domain and evaluate it on a target domain.

    Generalises `nested_layer_probe` and `nested_layer_probe_residualised` to the
    cross-domain case. All selection -- layer, `C`, scaler statistics, and the
    length residualiser when enabled -- is fit on source training rows only.

    Args:
        acts_src: Source activations (n_src, n_layers + 1, hidden).
        y_src: Source binary labels.
        len_src: Source prompt lengths in tokens.
        splits_src: Outer `(train_idx, test_idx)` pairs over the source; only
            the training halves are used.
        acts_tgt: Target activations.
        y_tgt: Target binary labels.
        len_tgt: Target prompt lengths in tokens.
        target_pool: Per-split index arrays into the target dataset.
        layers: Candidate layer indices.
        C_grid: Candidate regularisation strengths.
        inner_folds: Inner CV folds.
        seed: Base seed.
        residualise: Whether to residualise activations against prompt length,
            using coefficients fit on the source training fold and applied
            unchanged to the target.

    Returns:
        Dict with `auc`, `brier`, `layer`, `C` arrays of length `len(splits_src)`,
        plus `split_idx` / `split_p` over the target and `src_split_p` /
        `src_split_idx` holding the source held-out scores from the same fitted
        probes (Section 8 needs both sides).

    Raises:
        ValueError: If `target_pool` is shorter than `splits_src`.
    """
    if len(target_pool) < len(splits_src):
        raise ValueError("target_pool must have at least one resample per source split")

    aucs, briers, sel_layers, sel_C = [], [], [], []
    split_idx, split_p, src_idx, src_p = [], [], [], []

    for s, (tr, te) in enumerate(splits_src):
        rs = seed + s
        n_min = int(min(np.bincount(y_src[tr]).min(), inner_folds))
        skf = StratifiedKFold(n_splits=max(2, n_min), shuffle=True, random_state=rs)
        inner = list(skf.split(np.zeros(len(tr)), y_src[tr]))

        best = (-np.inf, layers[0], float(min(C_grid)))
        cache: Dict[int, Tuple[np.ndarray, Optional[Tuple[np.ndarray, np.ndarray]]]] = {}
        for layer in layers:
            Xtr = acts_src[tr][:, layer, :]
            coef = None
            if residualise:
                coef = fit_length_residualiser(Xtr, len_src[tr])
                Xtr = apply_length_residualiser(Xtr, len_src[tr], *coef)
            cache[layer] = (Xtr, coef)
            for C in C_grid:
                scores = []
                for itr, iva in inner:
                    probe = make_probe(C, CONFIG["pca_components"], rs)
                    probe.fit(Xtr[itr], y_src[tr][itr])
                    scores.append(_safe_auc(y_src[tr][iva], probe.predict_proba(Xtr[iva])[:, 1]))
                m = float(np.nanmean(scores))
                if m > best[0]:
                    best = (m, layer, float(C))

        _, layer_star, C_star = best
        Xtr, coef = cache[layer_star]
        probe = make_probe(C_star, CONFIG["pca_components"], rs)
        probe.fit(Xtr, y_src[tr])

        Xsrc_te = acts_src[te][:, layer_star, :]
        idx = np.asarray(target_pool[s])
        Xtgt = acts_tgt[idx][:, layer_star, :]
        if residualise:
            Xsrc_te = apply_length_residualiser(Xsrc_te, len_src[te], *coef)
            Xtgt = apply_length_residualiser(Xtgt, len_tgt[idx], *coef)

        p_tgt = probe.predict_proba(Xtgt)[:, 1]
        y_e = y_tgt[idx]
        aucs.append(_safe_auc(y_e, p_tgt))
        briers.append(float(brier_score_loss(y_e, p_tgt)) if len(np.unique(y_e)) > 1 else np.nan)
        sel_layers.append(layer_star)
        sel_C.append(C_star)
        split_idx.append(idx)
        split_p.append(p_tgt)
        src_idx.append(np.asarray(te))
        src_p.append(probe.predict_proba(Xsrc_te)[:, 1])

    return {
        "auc": np.asarray(aucs, dtype=float),
        "brier": np.asarray(briers, dtype=float),
        "layer": np.asarray(sel_layers, dtype=int),
        "C": np.asarray(sel_C, dtype=float),
        "split_idx": split_idx, "split_p": split_p,
        "src_split_idx": src_idx, "src_split_p": src_p,
    }


TARGET_POOL: Dict[str, List[np.ndarray]] = {
    ds: make_target_resamples(LABELS[ds]["y"], CONFIG["n_outer_splits"], CONFIG["seed"])
    for ds in CONFIG["datasets"]
}
print("target bootstrap resamples built:",
      {ds: (len(TARGET_POOL[ds]), len(TARGET_POOL[ds][0])) for ds in CONFIG["datasets"]})

> ⏱ **Slowest cell in the notebook.** Runs the full nested grid for every ordered
> (source, target) pair and for every feature family. At the defaults this is roughly
> `9 pairs x 50 splits x |layers| x |C| x inner_folds` fits for the probe alone. If you need it
> faster, first drop `n_outer_splits` to 20 and `layer_grid_stride` to 4 -- those cost CI width,
> not validity.

In [ ]:
FLAT_FEATURES: Dict[str, Dict[str, np.ndarray]] = {
    ds: {
        "length": PROMPT_LEN[ds].astype(np.float32).reshape(-1, 1),
        "embed": EMBEDS[ds],
    }
    for ds in CONFIG["datasets"]
}

E4: Dict[str, Dict[Tuple[str, str], Dict[str, Any]]] = {
    "probe": {}, "length": {}, "embed": {},
}
if CONFIG["e4_residualised_transfer"]:
    E4["probe_resid"] = {}

for src in CONFIG["datasets"]:
    for tgt in CONFIG["datasets"]:
        if src == tgt:
            # Diagonal: reuse the honest in-domain nested-CV results from E1/E2.
            E4["probe"][(src, tgt)] = E1[src]["probe"]
            E4["length"][(src, tgt)] = E1[src]["length"]
            E4["embed"][(src, tgt)] = E1[src]["embed"]
            if CONFIG["e4_residualised_transfer"]:
                E4["probe_resid"][(src, tgt)] = E2[src]
            print(f"[E4] {src} -> {tgt}: diagonal, reusing in-domain results")
            continue

        print(f"[E4] {src} -> {tgt}", flush=True)
        E4["probe"][(src, tgt)] = transfer_nested_probe(
            ACTS[src], LABELS[src]["y"], PROMPT_LEN[src], SPLITS[src],
            ACTS[tgt], LABELS[tgt]["y"], PROMPT_LEN[tgt], TARGET_POOL[tgt],
            LAYER_GRID, CONFIG["probe_C_grid"], CONFIG["inner_folds"], CONFIG["seed"],
            residualise=False,
        )
        if CONFIG["e4_residualised_transfer"]:
            E4["probe_resid"][(src, tgt)] = transfer_nested_probe(
                ACTS[src], LABELS[src]["y"], PROMPT_LEN[src], SPLITS[src],
                ACTS[tgt], LABELS[tgt]["y"], PROMPT_LEN[tgt], TARGET_POOL[tgt],
                LAYER_GRID, CONFIG["probe_C_grid"], CONFIG["inner_folds"], CONFIG["seed"],
                residualise=True,
            )
        for fam in ["length", "embed"]:
            E4[fam][(src, tgt)] = evaluate_feature_set(
                FLAT_FEATURES[src][fam], LABELS[src]["y"], SPLITS[src],
                CONFIG["probe_C_grid"], CONFIG["inner_folds"], CONFIG["seed"],
                X_test=FLAT_FEATURES[tgt][fam], y_test=LABELS[tgt]["y"],
                test_index_pool=TARGET_POOL[tgt],
            )
        print(f"   done {src} -> {tgt}", flush=True)

In [ ]:
def transfer_matrix(family: str, stat: str = "auc") -> pd.DataFrame:
    """Assemble a train-on-row / test-on-column matrix of a summary statistic.

    Args:
        family: Key into `E4`, e.g. "probe", "length", "embed".
        stat: Either "auc" or "brier".

    Returns:
        DataFrame whose rows are source datasets and columns target datasets,
        holding the mean of `stat` over splits.

    Raises:
        KeyError: If `family` was not computed.
    """
    if family not in E4:
        raise KeyError(f"family {family!r} not in E4; available: {list(E4)}")
    m = pd.DataFrame(index=CONFIG["datasets"], columns=CONFIG["datasets"], dtype=float)
    for (src, tgt), res in E4[family].items():
        m.loc[src, tgt] = float(np.nanmean(res[stat]))
    m.index.name = "train on"
    m.columns.name = "test on"
    return m


def transfer_matrix_ci(family: str) -> pd.DataFrame:
    """Assemble the same matrix as strings carrying the 95% interval.

    Args:
        family: Key into `E4`.

    Returns:
        DataFrame of `"mean [lo, hi]"` strings.

    Raises:
        KeyError: If `family` was not computed.
    """
    m = pd.DataFrame(index=CONFIG["datasets"], columns=CONFIG["datasets"], dtype=object)
    for (src, tgt), res in E4[family].items():
        mu, lo, hi = bootstrap_ci(res["auc"], CONFIG["ci_alpha"])
        m.loc[src, tgt] = f"{mu:.3f} [{lo:.3f}, {hi:.3f}]"
    m.index.name = "train on"
    m.columns.name = "test on"
    return m


for _fam in E4:
    print(f"\n===== E4 transfer AUC, family = {_fam} =====")
    display(transfer_matrix_ci(_fam))

In [ ]:
fams = list(E4.keys())
fig, axes = plt.subplots(1, len(fams), figsize=(4.3 * len(fams), 3.8))
axes = np.atleast_1d(axes)
for ax, fam in zip(axes, fams):
    m = transfer_matrix(fam).astype(float)
    im = ax.imshow(m.values, vmin=0.4, vmax=0.9, cmap="viridis")
    ax.set_xticks(range(len(m.columns))); ax.set_xticklabels(m.columns, rotation=30)
    ax.set_yticks(range(len(m.index))); ax.set_yticklabels(m.index)
    ax.set_xlabel("test on"); ax.set_ylabel("train on")
    ax.set_title(fam)
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            ax.text(j, i, f"{m.values[i, j]:.3f}", ha="center", va="center",
                    color="white" if m.values[i, j] < 0.72 else "black", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("E4: transfer AUC. Diagonal = held-out in-domain (E1); off-diagonal = zero target data.",
             y=1.06)
fig.tight_layout()
fig.savefig(FIG_DIR / "e4_transfer_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
rows = []
for fam in E4:
    for src in CONFIG["datasets"]:
        diag = float(np.nanmean(E4[fam][(src, src)]["auc"]))
        for tgt in CONFIG["datasets"]:
            if src == tgt:
                continue
            off = E4[fam][(src, tgt)]["auc"]
            in_domain_tgt = E4[fam][(tgt, tgt)]["auc"]
            m, lo, hi = bootstrap_ci(off, CONFIG["ci_alpha"])
            # Retention: how much of the target's own in-domain headroom above chance survives.
            ret = (m - 0.5) / max(float(np.nanmean(in_domain_tgt)) - 0.5, 1e-9)
            rows.append({
                "family": fam, "train": src, "test": tgt,
                "transfer_AUC": m, "lo": lo, "hi": hi,
                "source_in_domain_AUC": diag,
                "target_in_domain_AUC": float(np.nanmean(in_domain_tgt)),
                "retention_vs_target_in_domain": ret,
                "layer_mode": float(pd.Series(E4[fam][(src, tgt)]["layer"]).mode().iloc[0])
                if "layer" in E4[fam][(src, tgt)] else np.nan,
            })
E4_TABLE = pd.DataFrame(rows).set_index(["family", "train", "test"]).sort_index()
display(E4_TABLE)
print("retention_vs_target_in_domain = (transfer AUC - 0.5) / (target's own in-domain AUC - 0.5).")
print("1.0 means the transferred probe is as good as one trained on the target;")
print("0.0 means it is at chance; negative means it is anti-correlated on the target.")

### Learning curve: how much target data closes the gap

A transfer matrix says *whether* the gap exists. The operationally useful question is *what it
costs to close it* -- if 50 labelled target queries recover most of the in-domain performance, the
domain-shift problem is a minor annoyance; if it takes 400, the router has to be re-labelled per
domain and the "train once, deploy everywhere" story is gone.

**Protocol.** For each (source, target) pair and each budget `m`, three probes are compared on the
same held-out target test set:

- **source-only** (`m` ignored): the E4 transfer number, drawn as a horizontal reference.
- **source + m target**: source training rows pooled with `m` labelled target rows.
- **target-only, m rows**: the same `m` rows without any source data, which is what tells you
  whether the source data is helping at all or just diluting.

**Layer choice.** Fixed to the layer most often selected by the *source* in-domain nested CV.
Re-running full nested selection inside every point of the learning curve would multiply runtime by
`|layers| x |C| x inner_folds` for no additional insight, and using a target-selected layer would
leak. `C` is still chosen by inner CV on whatever the training set is at that point.

In [ ]:
def learning_curve(
    src: str,
    tgt: str,
    sizes: Sequence[int],
    n_reps: int,
    layer: int,
    C_grid: Sequence[float],
    inner_folds: int,
    seed: int,
) -> pd.DataFrame:
    """Measure how target-domain AUC improves as labelled target examples are added.

    Args:
        src: Source dataset name.
        tgt: Target dataset name.
        sizes: Numbers of labelled target examples to mix into training. A size
            of 0 gives the pure-transfer point; the target-only arm is undefined
            there and reported as NaN.
        n_reps: Repetitions per size; each redraws the target train/test split.
        layer: Fixed layer index, chosen from source-side information only.
        C_grid: Candidate regularisation strengths for inner CV.
        inner_folds: Inner CV folds.
        seed: Base seed.

    Returns:
        Long-form DataFrame with columns `m`, `rep`, `arm`, `auc`, where `arm`
        is one of "source+target", "target only", "source only".

    Raises:
        ValueError: If any size exceeds the available target training pool.
    """
    Xs = ACTS[src][:, layer, :]
    ys = LABELS[src]["y"]
    Xt = ACTS[tgt][:, layer, :]
    yt = LABELS[tgt]["y"]
    max_pool = int(len(yt) * (1 - CONFIG["outer_test_size"]))
    if max(sizes) > max_pool:
        raise ValueError(f"size {max(sizes)} exceeds target train pool of {max_pool}")

    rows = []
    for rep in range(n_reps):
        rs = seed + 7000 + rep
        sss = StratifiedShuffleSplit(
            n_splits=1, test_size=CONFIG["outer_test_size"], random_state=rs
        )
        pool_idx, test_idx = next(sss.split(np.zeros(len(yt)), yt))
        rng = np.random.default_rng(rs)
        src_tr = rng.choice(len(ys), size=int(len(ys) * (1 - CONFIG["outer_test_size"])),
                            replace=False)

        for m in sizes:
            take = pool_idx[:0] if m == 0 else rng.choice(pool_idx, size=m, replace=False)

            Xmix = np.vstack([Xs[src_tr], Xt[take]]) if m > 0 else Xs[src_tr]
            ymix = np.concatenate([ys[src_tr], yt[take]]) if m > 0 else ys[src_tr]
            C = select_C(Xmix, ymix, C_grid, inner_folds, rs)
            probe = make_probe(C, CONFIG["pca_components"], rs)
            probe.fit(Xmix, ymix)
            rows.append({"m": m, "rep": rep, "arm": "source+target",
                         "auc": _safe_auc(yt[test_idx],
                                          probe.predict_proba(Xt[test_idx])[:, 1])})

            if m > 0 and len(np.unique(yt[take])) > 1:
                C_t = select_C(Xt[take], yt[take], C_grid, inner_folds, rs)
                probe_t = make_probe(C_t, CONFIG["pca_components"], rs)
                probe_t.fit(Xt[take], yt[take])
                auc_t = _safe_auc(yt[test_idx], probe_t.predict_proba(Xt[test_idx])[:, 1])
            else:
                auc_t = float("nan")
            rows.append({"m": m, "rep": rep, "arm": "target only", "auc": auc_t})

    return pd.DataFrame(rows)


SOURCE_LAYER = {
    ds: int(pd.Series(E1[ds]["probe"]["layer"]).mode().iloc[0]) for ds in CONFIG["datasets"]
}
print("modal in-domain selected layer per dataset (used to fix the learning-curve layer):",
      SOURCE_LAYER)

In [ ]:
LEARNING: Dict[Tuple[str, str], pd.DataFrame] = {}
for src in CONFIG["datasets"]:
    for tgt in CONFIG["datasets"]:
        if src == tgt:
            continue
        # Budgets larger than the target's own training pool are dropped rather than
        # silently truncated, so a small dataset does not fake a large-m data point.
        pool = int(len(LABELS[tgt]["y"]) * (1 - CONFIG["outer_test_size"]))
        sizes = [m for m in CONFIG["learning_curve_sizes"] if m <= pool]
        if sizes != list(CONFIG["learning_curve_sizes"]):
            print(f"   note: target {tgt} pool is {pool}; using sizes {sizes}")
        print(f"[E4-LC] {src} -> {tgt}", flush=True)
        LEARNING[(src, tgt)] = learning_curve(
            src, tgt, sizes, CONFIG["learning_curve_reps"],
            SOURCE_LAYER[src], CONFIG["probe_C_grid"], CONFIG["inner_folds"], CONFIG["seed"],
        )
print("done")

In [ ]:
pairs = [(s, t) for s in CONFIG["datasets"] for t in CONFIG["datasets"] if s != t]
ncol = 3
nrow = int(np.ceil(len(pairs) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.6 * nrow), squeeze=False)
for ax, (src, tgt) in zip(axes.ravel(), pairs):
    df = LEARNING[(src, tgt)]
    for arm, colr in [("source+target", "#4c72b0"), ("target only", "#dd8452")]:
        g = df[df["arm"] == arm].groupby("m")["auc"]
        mu = g.mean()
        lo = g.quantile(CONFIG["ci_alpha"] / 2)
        hi = g.quantile(1 - CONFIG["ci_alpha"] / 2)
        ax.plot(mu.index, mu.values, marker="o", color=colr, label=arm)
        ax.fill_between(mu.index, lo.values, hi.values, color=colr, alpha=0.15)
    ax.axhline(float(np.nanmean(E4["probe"][(src, tgt)]["auc"])), color="grey",
               linestyle="--", label="source only (E4)")
    ax.axhline(float(np.nanmean(E4["probe"][(tgt, tgt)]["auc"])), color="black",
               linestyle=":", label=f"{tgt} in-domain (E1)")
    ax.axhline(0.5, color="red", linewidth=0.6)
    ax.set_title(f"train {src} -> test {tgt}")
    ax.set_xlabel("labelled target examples added (m)")
    ax.set_ylabel("target AUC")
    ax.legend(fontsize=6)
for ax in axes.ravel()[len(pairs):]:
    ax.axis("off")
fig.suptitle("E4: how many in-domain examples close the transfer gap", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "e4_learning_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
rows = []
for (src, tgt), df in LEARNING.items():
    transfer = float(np.nanmean(E4["probe"][(src, tgt)]["auc"]))
    in_dom = float(np.nanmean(E4["probe"][(tgt, tgt)]["auc"]))
    gap = in_dom - transfer
    for m in sorted(df["m"].unique()):
        mix = df[(df["arm"] == "source+target") & (df["m"] == m)]["auc"]
        tonly = df[(df["arm"] == "target only") & (df["m"] == m)]["auc"]
        mu, lo, hi = bootstrap_ci(mix.values, CONFIG["ci_alpha"])
        rows.append({
            "train": src, "test": tgt, "m": int(m),
            "source+target_AUC": mu, "lo": lo, "hi": hi,
            # m = 0 has no target-only arm at all, so this is NaN by construction there.
            "target_only_AUC": (float(np.nanmean(tonly.values))
                                if len(tonly) and not np.all(np.isnan(tonly.values)) else np.nan),
            "transfer_AUC(m=0)": transfer,
            "target_in_domain_AUC": in_dom,
            "gap_closed": (mu - transfer) / gap if abs(gap) > 1e-9 else np.nan,
        })
E4_LC_TABLE = pd.DataFrame(rows).set_index(["train", "test", "m"]).sort_index()
display(E4_LC_TABLE)
print("gap_closed = 1.0 means m target examples fully recovered in-domain performance.")
print("Values above 1.0 mean the mixed probe beat the target's own in-domain probe --")
print("possible when the source acts as regularisation, and worth reporting rather than clipping.")

## Section 8 - E5: threshold stability

An AUC is a property of a *ranking*. A deployed router does not deploy a ranking; it deploys **one
number**. You pick a threshold once -- on whatever validation data you had, which is in-domain by
definition -- and every query afterwards is compared to it. If the score distribution shifts under
domain change, that fixed threshold silently changes what it does: the escalation *rate* moves, so
the spend moves, even if the ranking quality were perfectly preserved.

This is a distinct failure mode from the one E4 measures, and it can occur even when transfer AUC
is high. A probe can rank target queries well and still push every score above the in-domain
threshold, escalating everything and paying strong-model prices on all traffic.

### Protocol

For each ordered pair (source, target), and for each of the 50 splits:

1. Split the source held-out fold in half. Choose `t*` on the first half as the quantile that
   produces the intended escalation rate `f*` (default 0.30). **This half is the only data the
   threshold ever sees.**
2. Apply `t*` unchanged to the second source half. The realised rate here should land near `f*`;
   this is the in-domain control, and it calibrates how much of any later deviation is just noise.
3. Apply the same `t*` unchanged to the target domain. Record the **realised escalation rate**, the
   **realised accuracy**, and the **realised cost**.
4. Compare against a router that was allowed to re-pick its threshold on the target to hit `f*`
   exactly. The difference between (3) and (4) is the cost of not re-calibrating -- which is
   precisely what a deployed system cannot do without target labels.

The plots put the fixed-threshold operating point on the target's own frontier, so you can see both
whether it drifted along the curve and whether it fell below it.

In [ ]:
def threshold_stability(
    src: str, tgt: str, res: Dict[str, Any], target_rate: float
) -> pd.DataFrame:
    """Measure what an in-domain-chosen threshold does once the domain moves.

    Args:
        src: Source dataset name.
        tgt: Target dataset name.
        res: A transfer result carrying `src_split_p` / `src_split_idx` (source
            held-out scores) and `split_p` / `split_idx` (target scores).
        target_rate: Intended escalation rate `f*` used to pick the threshold.

    Returns:
        Per-split DataFrame with the chosen threshold, the realised escalation
        rate in-domain and out-of-domain, the realised accuracy and cost under
        the fixed threshold, and the accuracy and cost a target-recalibrated
        threshold would have achieved at exactly `f*`.

    Raises:
        KeyError: If `res` lacks source-side per-split scores.
        ValueError: If `target_rate` is outside (0, 1).
    """
    if not 0.0 < target_rate < 1.0:
        raise ValueError(f"target_rate must be in (0, 1), got {target_rate}")
    for key in ("src_split_p", "src_split_idx", "split_p", "split_idx"):
        if key not in res:
            raise KeyError(f"transfer result is missing {key!r}")

    w_tgt = LABELS[tgt]["pass_rate"]
    s_tgt, _ = strong_accuracy(tgt)
    cw, cs = COSTS[tgt]["weak"], COSTS[tgt]["strong"]

    rows = []
    for s in range(len(res["split_p"])):
        sp = np.asarray(res["src_split_p"][s], dtype=float)
        rng = np.random.default_rng(CONFIG["seed"] + 9000 + s)
        perm = rng.permutation(len(sp))
        half = len(sp) // 2
        calib, check = sp[perm[:half]], sp[perm[half:]]

        t_star = float(np.quantile(calib, 1.0 - target_rate))
        realised_src = float((check >= t_star).mean())

        tp = np.asarray(res["split_p"][s], dtype=float)
        idx = np.asarray(res["split_idx"][s])
        esc = tp >= t_star
        realised_tgt = float(esc.mean())
        acc_fixed = float(np.where(esc, s_tgt[idx], w_tgt[idx]).mean())
        cost_fixed = float(np.where(esc, cs[idx], cw[idx]).mean())

        # Counterfactual: the same ranking, but re-thresholded on the target to hit f* exactly.
        t_recal = float(np.quantile(tp, 1.0 - target_rate))
        esc_r = tp >= t_recal
        acc_recal = float(np.where(esc_r, s_tgt[idx], w_tgt[idx]).mean())
        cost_recal = float(np.where(esc_r, cs[idx], cw[idx]).mean())

        rows.append({
            "split": s, "t_star": t_star, "t_recal": t_recal,
            "realised_rate_in_domain": realised_src,
            "realised_rate_target": realised_tgt,
            "rate_error": realised_tgt - target_rate,
            "acc_fixed_threshold": acc_fixed,
            "acc_recalibrated": acc_recal,
            "acc_delta": acc_fixed - acc_recal,
            "cost_fixed_threshold": cost_fixed,
            "cost_recalibrated": cost_recal,
            "cost_ratio": cost_fixed / max(cost_recal, 1e-12),
        })
    return pd.DataFrame(rows)


E5: Dict[Tuple[str, str], pd.DataFrame] = {}
for src in CONFIG["datasets"]:
    for tgt in CONFIG["datasets"]:
        if src == tgt:
            continue
        E5[(src, tgt)] = threshold_stability(
            src, tgt, E4["probe"][(src, tgt)], CONFIG["target_escalation_rate"]
        )
print(f"E5 computed for {len(E5)} ordered pairs at f* = {CONFIG['target_escalation_rate']}")

In [ ]:
rows = []
for (src, tgt), df in E5.items():
    def ci(col: str) -> Tuple[float, float, float]:
        return bootstrap_ci(df[col].values, CONFIG["ci_alpha"])
    r_m, r_lo, r_hi = ci("realised_rate_target")
    s_m, _, _ = ci("realised_rate_in_domain")
    a_m, a_lo, a_hi = ci("acc_delta")
    c_m, c_lo, c_hi = ci("cost_ratio")
    rows.append({
        "train": src, "test": tgt,
        "intended_rate": CONFIG["target_escalation_rate"],
        "realised_rate_in_domain": s_m,
        "realised_rate_target": r_m, "rate_lo": r_lo, "rate_hi": r_hi,
        "acc_fixed": float(df["acc_fixed_threshold"].mean()),
        "acc_recalibrated": float(df["acc_recalibrated"].mean()),
        "acc_delta": a_m, "acc_delta_lo": a_lo, "acc_delta_hi": a_hi,
        "cost_ratio_fixed_vs_recal": c_m, "cost_lo": c_lo, "cost_hi": c_hi,
    })
E5_TABLE = pd.DataFrame(rows).set_index(["train", "test"]).sort_index()
display(E5_TABLE)
print("realised_rate_in_domain is the control: it should sit near intended_rate.")
print("realised_rate_target is what the SAME threshold actually does after the shift.")
print("cost_ratio_fixed_vs_recal > 1 means the drift made the deployed router more expensive")
print("than intended; < 1 means it quietly stopped escalating and saved money by under-serving.")

In [ ]:
pairs = list(E5.keys())
ncol = 3
nrow = int(np.ceil(len(pairs) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.8 * ncol, 3.8 * nrow), squeeze=False)
for ax, (src, tgt) in zip(axes.ravel(), pairs):
    fr = FRONTIERS[tgt]
    x = fr["fractions"]
    ax.plot(x, fr["acc"]["probe"].mean(0), color="#4c72b0",
            label=f"{tgt} frontier (probe trained in-domain)")
    ax.plot(x, fr["oracle_acc"].mean(0), color="black", linewidth=1, label="oracle")
    ax.plot(x, fr["random_acc"].mean(0), color="grey", linestyle="--", label="random")

    df = E5[(src, tgt)]
    ax.scatter([CONFIG["target_escalation_rate"]], [df["acc_recalibrated"].mean()],
               marker="s", s=60, color="green", zorder=5,
               label=f"recalibrated on target (f*={CONFIG['target_escalation_rate']})")
    ax.scatter([df["realised_rate_target"].mean()], [df["acc_fixed_threshold"].mean()],
               marker="X", s=90, color="crimson", zorder=6,
               label="FIXED in-domain threshold, deployed")
    ax.errorbar([df["realised_rate_target"].mean()], [df["acc_fixed_threshold"].mean()],
                xerr=[[df["realised_rate_target"].mean()
                       - np.percentile(df["realised_rate_target"], 2.5)],
                      [np.percentile(df["realised_rate_target"], 97.5)
                       - df["realised_rate_target"].mean()]],
                fmt="none", ecolor="crimson", capsize=3)
    ax.axvline(CONFIG["target_escalation_rate"], color="green", linewidth=0.6, linestyle=":")
    ax.set_title(f"threshold from {src}, deployed on {tgt}")
    ax.set_xlabel("escalation fraction")
    ax.set_ylabel("expected accuracy")
    ax.legend(fontsize=6, loc="lower right")
for ax in axes.ravel()[len(pairs):]:
    ax.axis("off")
fig.suptitle("E5: where an in-domain threshold lands on the out-of-domain frontier", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "e5_threshold_on_ood_frontier.png", dpi=150, bbox_inches="tight")
plt.show()

### Why the threshold drifts: the score distributions themselves

The mechanism behind any drift above is visible directly. If the target score distribution is
shifted relative to the source, a fixed cut through both lands at different quantiles, and the
escalation rate moves by exactly that quantile difference -- independently of how well the scores
*rank* target queries. Ranking quality (E4) and location (E5) are separate properties, and only the
first is what an AUC reports.

In [ ]:
fig, axes = plt.subplots(nrow, ncol, figsize=(4.8 * ncol, 3.4 * nrow), squeeze=False)
for ax, (src, tgt) in zip(axes.ravel(), pairs):
    res = E4["probe"][(src, tgt)]
    src_scores = np.concatenate(res["src_split_p"])
    tgt_scores = np.concatenate(res["split_p"])
    bins = np.linspace(0, 1, 41)
    ax.hist(src_scores, bins=bins, density=True, alpha=0.5, label=f"source held-out ({src})")
    ax.hist(tgt_scores, bins=bins, density=True, alpha=0.5, label=f"target ({tgt})")
    ax.axvline(E5[(src, tgt)]["t_star"].mean(), color="crimson", linestyle="--",
               label="mean t* (chosen in-domain)")
    ax.set_title(f"{src} -> {tgt}")
    ax.set_xlabel("predicted P(weak model fails)")
    ax.set_ylabel("density")
    ax.legend(fontsize=6)
for ax in axes.ravel()[len(pairs):]:
    ax.axis("off")
fig.suptitle("E5: score distributions on either side of the shift, with the deployed cut", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "e5_score_shift.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 3.4 * nrow), squeeze=False)
for ax, (src, tgt) in zip(axes.ravel(), pairs):
    res = E4["probe"][(src, tgt)]
    y_pool = np.concatenate([LABELS[tgt]["y"][i] for i in res["split_idx"]])
    p_pool = np.concatenate(res["split_p"])
    plot_reliability(ax, y_pool, p_pool, CONFIG["calibration_bins"], f"{src} -> {tgt}")
    yd = E1[tgt]["probe"]
    plot_reliability(ax, yd["pooled_y"], yd["pooled_p"], CONFIG["calibration_bins"],
                     f"{tgt} in-domain")
    ax.plot([0, 1], [0, 1], "k--", linewidth=1)
    ax.set_xlabel("predicted P(fail)"); ax.set_ylabel("observed failure fraction")
    ax.set_title(f"{src} -> {tgt}")
    ax.legend(fontsize=6)
for ax in axes.ravel()[len(pairs):]:
    ax.axis("off")
fig.suptitle("E5: calibration under domain shift -- a mis-located curve is exactly what breaks\n"
             "a fixed threshold, even when the ranking survives", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "e5_calibration_shift.png", dpi=150, bbox_inches="tight")
plt.show()

## Section 9 - Results summary

One claim per subsection. Each holds **the artefact that would support it** and an explicit `TODO`
where the sentence goes. The prose is deliberately unwritten: none of these numbers exist until the
notebook is run, and writing a conclusion before seeing the number is how a project talks itself
into a result.

Fill each `TODO` in only after reading the table or figure directly above it, and if a number does
not support the claim, **rewrite the claim**, not the number.

### Claim 1 - the pre-generation signal exists in-domain (replication)

> **TODO.** State, per dataset, the nested-CV AUC of the activation probe with its 95% interval,
> and whether that interval excludes 0.5. Say which layer was selected most often and whether the
> layer curve peaks early, mid or late. If any dataset's interval includes 0.5, say so plainly --
> this is a replication, and a failed replication on one domain is a result.
>
> **TODO.** Note the label-quality check from Section 2 here: if any dataset was flagged, every
> claim about it is provisional.

In [ ]:
print("=" * 78)
print("CLAIM 1 SUPPORT: in-domain nested-CV router comparison")
print("=" * 78)
for ds in CONFIG["datasets"]:
    print(f"\n--- {ds} ---")
    display(E1_TABLES[ds])
print("\n--- label quality (Section 2) ---")
display(quality)
print("\n--- layer curves (descriptive; the max is NOT the headline) ---")
for ds in CONFIG["datasets"]:
    print(f"\n{ds}: peak layer = {int(LAYER_CURVES[ds]['auc_mean'].idxmax())}, "
          f"peak AUC = {LAYER_CURVES[ds]['auc_mean'].max():.4f}  |  "
          f"nested-CV AUC = {np.nanmean(E1[ds]['probe']['auc']):.4f}  |  "
          f"modal selected layer = {int(pd.Series(E1[ds]['probe']['layer']).mode().iloc[0])}")

### Claim 2 - the signal is not just prompt length

> **TODO.** Report `d(resid - length)` per dataset with its interval. State whether the
> length-residualised probe still beats the length-only baseline, and by how much. State the drop
> from raw to residualised, which is the size of the confound.
>
> **TODO.** If the residualised advantage is small or its interval crosses zero on a dataset, say
> that the E1 result on that dataset is substantially a length effect. Remember the residualised
> figure is a lower bound (Section 5 note), and say that too.

In [ ]:
print("=" * 78)
print("CLAIM 2 SUPPORT: length residualisation")
print("=" * 78)
display(E2_TABLE)
display(length_exposure)

### Claim 3 - routing on the signal buys accuracy at a given budget

> **TODO.** At 20 / 30 / 50% escalation, state the fraction of oracle gain the probe captures per
> dataset, with intervals, and compare it against the length and embedding routers at the same
> budget.
>
> **TODO.** Say explicitly whether the probe beats post-hoc confidence, and remind the reader that
> post-hoc confidence is being charged for a weak generation it has already performed -- so a tie
> on accuracy is a loss on cost.
>
> **TODO.** Restate the cost-model caveat: the cost axis uses median output length, so it is
> relative, not exact.

In [ ]:
print("=" * 78)
print("CLAIM 3 SUPPORT: cost-quality frontier at fixed escalation budgets")
print("=" * 78)
display(E3_TABLE)
for ds in CONFIG["datasets"]:
    fr = FRONTIERS[ds]
    tag = "measured" if fr["strong_measured"] else f"FALLBACK {FALLBACK_STRONG_ACC}"
    print(f"{ds:6s}  weak-only={fr['weak_only']:.4f}  strong-only={fr['strong_only']:.4f} ({tag})  "
          f"headroom={fr['strong_only'] - fr['weak_only']:+.4f}")

### Claim 4 (contribution) - what happens to the signal under domain shift

> **TODO.** Read the transfer matrix off the diagonal. For each ordered pair, state the transfer
> AUC and the retention relative to the target's own in-domain probe. Group the pairs into those
> that transfer and those that do not, and say whether the pattern is symmetric.
>
> **TODO.** Compare the probe's matrix to the length and embedding matrices. If the probe's
> off-diagonal is no better than the embedding's, the probe is carrying topic difficulty rather
> than model-specific self-knowledge, and the claim has to be weakened to exactly that.
>
> **TODO.** State whether the layer selected on the source is the same layer the target would have
> chosen -- a mismatch is a mechanism for the drop and is visible in `layer_mode`.

In [ ]:
print("=" * 78)
print("CLAIM 4 SUPPORT: transfer matrices")
print("=" * 78)
for fam in E4:
    print(f"\n--- family: {fam} ---")
    display(transfer_matrix_ci(fam))
print("\n--- retention and selected layers ---")
display(E4_TABLE)

### Claim 5 (contribution) - how much target data closes the gap

> **TODO.** For each pair, state the smallest `m` at which `gap_closed` reaches roughly 0.5 and
> roughly 0.9, or say that the curve plateaus below that within the budgets tested.
>
> **TODO.** Compare the `source+target` arm against the `target only` arm. If they coincide, the
> source data is contributing nothing and the honest recommendation is to label target data and
> discard the source probe entirely.
>
> **TODO.** Convert the answer into a deployment sentence: "adapting this router to a new domain
> costs about N labelled queries, which is N x k weak-model generations."

In [ ]:
print("=" * 78)
print("CLAIM 5 SUPPORT: learning curve")
print("=" * 78)
display(E4_LC_TABLE)
print("\nBudget reminder: each labelled target example costs k =",
      CONFIG["k_samples"], "weak-model generations to label.")

### Claim 6 (contribution) - a fixed threshold does not survive the shift

> **TODO.** State the intended escalation rate, the realised in-domain rate (the control) and the
> realised target rate for each pair. Quantify the drift in percentage points.
>
> **TODO.** Translate the drift into money using `cost_ratio_fixed_vs_recal`: a router that was
> budgeted for 30% escalation and actually escalates X% costs Y times what was planned.
>
> **TODO.** Report `acc_delta` -- how much accuracy the fixed threshold gives up relative to a
> target-recalibrated one. Distinguish the two failure modes: drifting *along* the frontier (a
> budget failure, accuracy roughly preserved) versus falling *below* it (a ranking failure).
>
> **TODO.** Connect to the calibration figures: state whether the transferred probe is
> mis-calibrated on the target and in which direction.

In [ ]:
print("=" * 78)
print("CLAIM 6 SUPPORT: threshold stability under shift")
print("=" * 78)
display(E5_TABLE)
print("\n--- calibration summary (Brier, lower is better) ---")
rows = []
for ds in CONFIG["datasets"]:
    for name in ["length", "embed", "posthoc", "probe"]:
        m, lo, hi = bootstrap_ci(E1[ds][name]["brier"], CONFIG["ci_alpha"])
        rows.append({"setting": f"in-domain {ds}", "router": ROUTER_LABEL[name],
                     "Brier": m, "lo": lo, "hi": hi})
for (src, tgt), res in E4["probe"].items():
    if src == tgt:
        continue
    m, lo, hi = bootstrap_ci(res["brier"], CONFIG["ci_alpha"])
    rows.append({"setting": f"transfer {src} -> {tgt}", "router": ROUTER_LABEL["probe"],
                 "Brier": m, "lo": lo, "hi": hi})
CALIBRATION_TABLE = pd.DataFrame(rows).set_index(["setting", "router"]).sort_index()
display(CALIBRATION_TABLE)

### Claim 7 - the overall verdict

> **TODO.** One paragraph. Does pre-generation routing on a 0.5B model work well enough to deploy,
> and does it survive domain shift? Answer both halves separately; they can have different answers,
> and the interesting outcome for this project is precisely the case where the first is yes and the
> second is no.
>
> **TODO.** If the answer to the second half is negative, state the practical implication: routers
> must be fit per domain, and the labelling cost from Claim 5 is the price of entry.
>
> **TODO.** Name the single result you would most want to see replicated at a larger weak model
> before believing any of this generalises.

In [ ]:
ARTEFACTS = {
    "e1_router_tables": E1_TABLES,
    "e1_layer_curves": LAYER_CURVES,
    "e2_residualisation": E2_TABLE,
    "e3_frontier": E3_TABLE,
    "e4_transfer": E4_TABLE,
    "e4_learning_curve": E4_LC_TABLE,
    "e5_threshold_stability": E5_TABLE,
    "calibration": CALIBRATION_TABLE,
    "label_quality": quality,
    "length_exposure": length_exposure,
}

out_dir = Path("results")
out_dir.mkdir(exist_ok=True)
for name, obj in ARTEFACTS.items():
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(out_dir / f"{name}.csv")
    else:
        for key, df in obj.items():
            df.to_csv(out_dir / f"{name}__{key}.csv")

write_json(out_dir / "config_used.json", CONFIG)
print(f"wrote {len(list(out_dir.glob('*')))} files to {out_dir.resolve()}")
print("Figures are in", FIG_DIR.resolve())
print("\nEvery number in those files came from this run. Nothing here is illustrative.")

## Section 10 - Limitations

Written out in full, because most of these bound the conclusions rather than merely decorating them.

**1. One weak model, one size.** Every result is for `Qwen/Qwen2.5-0.5B`. Whether hidden states
encode impending failure is plausibly a function of scale and of instruction tuning, and a 0.5B
base model is at the far end of both axes. Nothing here licenses a claim about 7B or 70B models, or
about instruction-tuned variants of the same size. `MODEL_NAME` is a single constant precisely so
this can be re-run, and re-running it on a second model is the first thing that should happen.

**2. One strong model, evaluated out of its native format.** The strong model receives the same
few-shot base-model prompt as the weak one, so that the query is held fixed across the routing
decision. An instruct-tuned model prompted without its chat template underperforms, which means the
strong-model accuracy in E3 and E5 is an **under-estimate**, the headroom is compressed, and the
oracle ceiling is lower than it should be. The direction of that bias is known; the magnitude is
not measured here.

**3. Three datasets is a small sample of "domains".** The transfer matrix has six off-diagonal
cells. Those cells are entangled with dataset-specific artefacts -- prompt format, answer format,
output length, grader strictness -- that are not "domain" in any interesting sense. A finding that
GSM8K does not transfer to MMLU may be a finding about numeric-answer versus letter-answer prompts.
Distinguishing the two would require several datasets per skill, which this design does not have.

**4. Grading is not correctness.** GSM8K is graded by last-number match, which credits a lucky
number and penalises a correct answer stated in words. MMLU is graded by first-letter match, which
is sensitive to how the base model formats its reply and can penalise a model that knows the answer
but formats it badly. MBPP is graded by executing three assertions, which is the strongest of the
three but still accepts code that passes them and fails everything else. All three graders will
therefore mislabel some queries, and label noise depresses every AUC by an unknown amount that
differs per dataset -- which matters, since cross-dataset comparison is the point of E4.

**5. Pass-rate labels depend on `k`, temperature and the threshold.** The binary label is a
threshold on an 8-sample estimate of a probability. At `k = 8` the pass rate has a standard error of
up to ~0.18, so queries near the 0.5 threshold are labelled close to arbitrarily. The raw
generations are cached specifically so this can be probed by re-thresholding, but the primary
analysis uses one threshold and inherits its noise.

**6. Linear residualisation removes only linear length effects.** Section 5 already states this:
the residualised AUC over-estimates the length-free signal if the dependence on length is curved,
and under-estimates the genuine difficulty signal whenever difficulty and length are legitimately
correlated. Both bounds are reported; neither is the truth.

**7. Prompt length is one confound of several.** Vocabulary rarity, presence of digits, question
type and topic all covary with difficulty and are all encoded in the hidden state. Only length is
controlled. The query-embedding router (4) partially addresses topic, but a probe that beats it
could still be exploiting some fourth thing.

**8. The cost model is a median-length approximation.** Output cost is charged at the dataset median
output length rather than the realised length, and the token prices are placeholders whose only
meaningful content is the weak/strong ratio. The cost axis supports statements of the form "router A
dominates router B", not "this saves $X per million queries". Real serving costs also include
batching effects, KV-cache reuse and latency SLOs, none of which appear here.

**9. The intervals are repeated-holdout intervals, not bootstrap intervals over a fixed test set.**
They summarise variation across 50 stratified resamples of a fixed dataset of ~1000 examples. They
therefore describe uncertainty about *this data*, and they do **not** capture uncertainty about the
choice of dataset, prompt format, or grader -- which, per limitations 3 and 4, is likely the larger
source of error.

**10. The nested CV is honest about layer and `C`, and not about everything else.** Pooling
strategy (last token), feature type (raw hidden state), probe family (L2 logistic regression), and
the label definition were all fixed by hand after looking at the problem, not selected inside the
CV. The reported intervals do not account for that outer layer of researcher choice.

**11. Residualisation in E2 sees the inner validation folds.** The length residualiser is fit on the
whole outer training fold and the inner CV then runs on the residualised result. This cannot leak
into the outer test score, but it does mildly optimise the *selection* of layer and `C`. The effect
is expected to be small; it is not zero.

**12. E5 measures threshold drift, not threshold repair.** The comparison is between a fixed
in-domain threshold and one recalibrated on target labels. A deployed system without target labels
has neither option available and would need an unsupervised recalibration scheme -- quantile
matching on unlabelled target scores, for instance. That is the obvious next experiment and is not
run here.

**13. Escalation is modelled as a binary, one-shot decision.** No cascade of more than two models,
no retry-on-the-weak-model, no partial generation with an abort, no per-query budget. Real routing
systems use all of these, and a two-model one-shot cascade is the easiest possible version of the
problem.

**14. Expected accuracy is used in place of realised accuracy on the frontier.** Per-query pass
rates enter the frontier rather than sampled outcomes. This removes Monte-Carlo noise that has
nothing to do with the router, but it also means the frontier describes the *expected* behaviour of
a fleet of queries rather than any particular run.

**15. No statistical correction for multiple comparisons.** The notebook reports on the order of a
hundred intervals across six routers, three datasets, nine transfer cells and several escalation
points. Some will exclude 0.5 or 0 by chance. Treat the pre-registered comparisons -- probe versus
length, and diagonal versus off-diagonal -- as the confirmatory ones, and everything else as
exploratory.